# Urban Ride Time Prediction — Phase 1: Data Cleaning



## Imports

In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List,Tuple , Optional
import re

import numpy as np
import pandas as pd


try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [2]:
TARGET_COL = "elapsed_seconds"

EXPECTED_COLUMNS = [
    "record_key",
    "carrier_code",
    "start_time",
    "end_time",
    "elapsed_seconds",
    "rider_total",
    "start_xcoord",
    "start_ycoord",
    "end_xcoord",
    "end_ycoord",
    "save_forward_marker",
]

TIME_COLS = ["start_time", "end_time"]

NUMERIC_COLS = [
    "elapsed_seconds",
    "rider_total",
    "start_xcoord",
    "start_ycoord",
    "end_xcoord",
    "end_ycoord",
]

COORD_COLS = ["start_xcoord", "start_ycoord", "end_xcoord", "end_ycoord"]
X_COLS = ["start_xcoord", "end_xcoord"]
Y_COLS = ["start_ycoord", "end_ycoord"]

REMOVAL_COUNT_KEYS = [
    "missing_target_removed",
    "exact_duplicate_rows_removed",
    "duplicate_record_keys_removed",
    "invalid_duration_removed",
    "duration_outlier_removed",
    "missing_start_time_removed",
    "invalid_time_order_removed",
    "duration_mismatch_removed",
    "invalid_rider_total_removed",
    "invalid_coordinates_removed",
    "unrealistic_speed_or_distance_removed",
]


@dataclass
class CleaningConfig:
    input_file: str = "student_version.csv"
    cleaned_output_file: str = "cleaned_student_version.csv"
    report_csv_file: str = "cleaning_report.csv"
    report_txt_file: str = "cleaning_report.txt"
    flagged_output_file: str = "flagged_outliers.csv"

    min_valid_duration_seconds: float = 10.0
    duration_upper_percentile: float = 0.999
    min_rows_for_duration_outlier_rule: int = 100

    time_mismatch_fixed_tolerance_seconds: float = 300.0
    time_mismatch_relative_tolerance: float = 0.05

    max_reasonable_rider_total: float = 20.0

    zero_coordinate_tolerance: float = 1e-12


    max_reasonable_speed_kmh: float = 160.0
    max_reasonable_trip_distance_km: float = 300.0
    min_distance_for_speed_check_km: float = 0.05
    projected_distance_upper_percentile: float = 0.999

    serious_missing_pct: float = 30.0
    rare_category_min_count: int = 5

In [3]:

def normalized_column_name(col:str)-> str:
    
    text = str(col).strip().lower()
    text = text.replace('%','percent')
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")


    if text == '':
        text = 'unnamed_column'
        
    return text

def normalized_column_names(df:pd.DataFrame) -> Tuple[pd.DataFrame,Dict[str,str]]:
    
    
    used_names : Dict[str,int] = {}
    mapping: Dict[str,str] = {}
    new_columns : List[str] = []
    
    
    for raw_col in df.columns:
        base_name = normalized_column_name(raw_col)
        
        if base_name not in used_names:
            used_names[base_name] = 1
            final_name = base_name
        else:
            used_names[base_name] += 1
            final_name = f'{base_name}_{used_names[base_name]}'
            
        mapping[str(raw_col)] = final_name
        new_columns.append(final_name)
        
    df = df.copy()
    df.columns = new_columns
    
    return df,mapping


def add_report(report: List[Dict], step: str, metric: str, value, details: str = "") -> None:
    report.append({
        'step':step,
        'metric':metric,
        'value':value,
        'details':details
    })



def increment_count(removal_counts: Dict[str,int],key:str,amount:int)-> None:
    removal_counts[key] = int(removal_counts.get(key,0)) +int(amount)
    

def collect_flagged_rows(flagged_parts,df,mask,reason,action):
    if mask is None or len(mask) == 0 or int(mask.sum()) == 0:
        return
    
    part = df.loc[mask].copy()
    part['flag_reason'] = reason
    part['flag_action'] = action
    flagged_parts.append(part)
    
    
def remove_rows(
    df : pd.DataFrame,
    mask : pd.Series,
    reason:str,
    removal_key: str,
    removal_counts: Dict[str,int],
    flagged_parts: List[pd.DataFrame],
)-> pd.DataFrame:
    
    mask = mask.fillna(False).astype(bool)
    
    n_removed = int(mask.sum())
    n_total = len(df)
    
    collect_flagged_rows(
        flagged_parts=flagged_parts,
        df= df,
        mask = mask,
        reason= reason,
        action='removed'
    )
    
    if n_total > 0 and n_removed >= n_total:
        raise ValueError(
            f'This cleaning rule would remove all rows : {reason}'
            f'Rows  to remove :{n_removed:,}/ {n_total:,}'
            'this rule is too strict change it to flag-only or relax the threshold'
        )
    increment_count(removal_counts, removal_key, n_removed)

    if n_removed > 0 :
        print(f'Removed {n_removed:,} rows:{reason}')
    
    return df.loc[~mask].copy()


def initialize_removal_counts()-> Dict[str,int]:
    return {key:0 for key in REMOVAL_COUNT_KEYS}

def existing_columns(df: pd.DataFrame, cols: List[str]) -> List[str]:
    return [col for col in cols if col in df.columns]

def remove_helper_columns(df:pd.DataFrame) -> pd.DataFrame:
    
    helper_cols = [col for col in df.columns if col.startswith('_')]
    return df.drop(columns=helper_cols,errors='ignore').copy()


def safe_percent(value:float,denominator:float)-> float:
    if denominator == 0:
        return 0.0
    return 100.0*float(value)/float(denominator)

In [4]:
def validate_expected_columns(df:pd.DataFrame,report:List[Dict]) -> None:
    missing_expected = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    present_expected = [col for col in EXPECTED_COLUMNS  if col in df.columns]
    
    
    add_report(
        report,
        step='Step 1: Load and inspect data',
        metric= 'expected_columns_present',
        value = len(present_expected),
        details = ','.join(present_expected),
    )
        
    add_report(
        report,
        step='Step 1: Load and inspect data',
        metric= 'expected_columns_mising',
        value = len(missing_expected),
        details = ','.join(missing_expected) if missing_expected else 'None',
    )
    
    if missing_expected:
        print('Warning : Some expected columns are missing')
        print(missing_expected)
        
    if TARGET_COL not in df.columns:
        raise ValueError(
            f'Required target column {TARGET_COL} is missing  after column normalization'
            "please check the dataset column names  "
        )
        
def load_and_inspect_data(config:CleaningConfig,report:List[Dict])-> Tuple[pd.DataFrame,int]:
    
    print("Loading dataset...")
    
    input_path = Path(config.input_file)
    
    if not input_path.exists():
        raise FileNotFoundError(f'Input file not found {input_path.resolve()}')
    
    
    df_raw = pd.read_csv(input_path)
    original_rows = len(df_raw)
    
    
    print('Raw Shape:',df_raw.shape)
    print('Raw Columns:')
    print(list(df_raw.columns))
    print('\nRAw dtypes: ')
    display(df_raw.dtypes)
    print('\n First rows before normalization')
    display(df_raw.head())
    
    df,column_mapping = normalized_column_names(df_raw)
    
    print('\n Column normalization mapping:')
    display(pd.DataFrame({'raw_column':list(column_mapping.keys()), 'normalized_column':list(column_mapping.values())}))
    
    print("\nShape after column normalization:", df.shape)
    print("Normalized columns:")
    print(list(df.columns))
    print("\nDtypes after column normalization:")
    display(df.dtypes)
    print("\nFirst rows after normalization:")
    display(df.head())

    validate_expected_columns(df, report)

    add_report(report, "Step 1: Load and inspect data", "original_rows", original_rows)
    add_report(report, "Step 1: Load and inspect data", "original_columns_count", len(df_raw.columns))
    add_report(report, "Step 1: Load and inspect data", "normalized_columns_count", len(df.columns))

    return df, original_rows

In [5]:
def analyze_missing_values(df: pd.DataFrame,config:CleaningConfig,report: List[Dict]) -> pd.DataFrame:
    
    missing = pd.DataFrame({
        'missing_count':df.isna().sum(),
        'missing_pct':df.isna().mean()*100.0,
    }).sort_values('missing_pct',ascending=False)
    
    print('Missing value analysis')
    display(missing)
    
    
    serious = missing[missing['missing_pct']>= config.serious_missing_pct]
    if len(serious)>0:
        print(f'\n Columns with serious missingess >= {config.serious_missing_pct:.1f}%:')
        display(serious)
    else:
        print(f'\n No columns exceed the serious missing threshold of {config.serious_missing_pct:.1f}%')
        
    
    for col , row in missing.iterrows():
        add_report(
            report,
            step='step 2 : Missing value analysis',
            metric= f'missing_{col}',
            value = int(row['missing_count']),
            details=f'{row['missing_pct']:.2f} % missing'
        )
    return missing


In [6]:
def remove_duplicates(
    df:  pd.DataFrame,
    report: List[Dict],
    removal_counts : Dict[str,int],
    flagged_parts: List[pd.DataFrame],
)-> pd.DataFrame:
    
    step = 'Step 3 : Duplicate detection'
    
    exact_dup_mask = df.duplicated(keep='first')
    n_exact = int(exact_dup_mask.sum())
    add_report(report,step,'exact_duplicate_rows_found',n_exact)
    
    
    df = remove_rows(
        df,
        exact_dup_mask,
        reason='Exact duplication row',
        removal_key = 'exact_duplicate_rows_removed',
        removal_counts=removal_counts,
        flagged_parts=flagged_parts,
    )
    
    
    if 'record_key' not in df.columns:
        add_report(report,step,'duplicate_record_key_check','slipped','record_key column is  missing')
        print("Duplicate record_key check skipped because 'record_key' is missing")
        return df

    non_missing_key_mask = df['record_key'].notna()
    duplicate_key_count = int(df.loc[non_missing_key_mask,'record_key'].duplicated(keep=False).sum())
    
    add_report(report,step,'duplicate_record_key_row_found',duplicate_key_count)
    
    
    if duplicate_key_count == 0:
        print("No duplicate record_key values found after exact duplicate removal.")
        return df 
    
    temp = df.loc[non_missing_key_mask].copy()
    temp['_missing_count_for_duplicate_choice'] = temp.isna().sum(axis=1)
    temp = temp.sort_values(['record_key','_missing_count_for_duplicate_choice'],kind='mergesort')
    
    keep_indices = temp.drop_duplicates(subset=['record_key'],keep='first').index
    remove_indices = temp.index.difference(keep_indices)
    
    
    duplicate_key_remove_mask = pd.Series(False,index=df.index)
    duplicate_key_remove_mask.loc[remove_indices] = True
    
    
    df = remove_rows(
        df,
        duplicate_key_remove_mask,
        
        reason='Duplicate record_key ,kept the rows with the fewest missing values',
        removal_key = 'duplicate_record_keys_removed',
        removal_counts=removal_counts,
        flagged_parts= flagged_parts,
    )
    
    df = df.drop(columns=['_missing_count_for_duplicate_choice'],errors='ignore')
    return df

In [7]:
def convert_data_types(
    df: pd.DataFrame,
    report: List[Dict],
    flagged_parts : List[pd.DataFrame],
)-> pd.DataFrame:
    
    step = 'Step 4: Data type correction'
    df = df.copy()
    
    
    for col in TIME_COLS:
        if col not in df.columns:
            add_report(report,step,f'{col}_datetime_conversion','skipped','column missing')
            continue
        
        original = df[col].copy()
        non_missing_before = original.notna()
        parsed = pd.to_datetime(original,errors='coerce')
        invalid_mask = non_missing_before & parsed.isna()
        
        collect_flagged_rows(
            flagged_parts,
            df,
            invalid_mask,
            reason = f'Invalid datetime  value in {col}',
            action='flagged_for_later_handeling',
        )
        df[col] = parsed
        add_report(report,step,f'Invalid_datatime_values_{col}',int(invalid_mask.sum()))
        
    for col in NUMERIC_COLS:
        if col not in df.columns:
            add_report(report,step,f'{col}_numeric_coversion','skipped','column missing')
            continue
        
        original = df[col].copy()
        non_missing_before = original.notna()
        
    return df
        

In [57]:
def print_duration_statistics(df: pd.DataFrame,config: CleaningConfig,report:List[Dict])-> Optional[float]:
    
    step = 'Step 5: Target cleaning'
    
    if TARGET_COL not in df.columns or df[TARGET_COL].dropna().empty:
        add_report(report,step,'duration_statistics','skipped','target missing or empty')
        return None
    
    duration = df[TARGET_COL].dropna()
    
    print('Duration descriptive statistics:')
    display(duration.describe())
    
    
    quantiles = duration.quantile([0.01,0.05,0.25,0.50,0.75,0.95,0.99,config.duration_upper_percentile])
    print('\n Duration quantiles')
    display(quantiles)
    
    
    add_report(report,step,'duration_min',float(duration.min()))
    add_report(report,step,'duration_max',float(duration.max()))
    add_report(report,step,'duration_median',float(duration.median()))
    
    
    if len(duration)< config.min_rows_for_duration_outlier_rule:
        
        add_report(
            report,
            step,
            'duration_outlier_rule',
            'skipped',
            f'Only {len(duration)} rows available; threshold requires at least {config.min_rows_for_duration_outlier_rule} rows',
        )
        return None
    
    q1 = duration.quantile(0.25)
    q3 = duration.quantile(0.75)
    iqr = q3 - q1 
    iqr_upper = q3 + 3.0*iqr
    pct_upper = duration.quantile(config.duration_upper_percentile)
    
    
    upper_limit = max(float(iqr_upper),float(pct_upper))
    
    
    add_report(
        report,
        step,
        'duration_upper_limit_seconds',
        upper_limit,
        f'max(Q3 + 3*IRQ = {iqr_upper:.2f}, q {config.duration_upper_percentile: .3f} = {pct_upper: .2f})',
    )
    
    return upper_limit


def clean_target(
    df : pd.DataFrame,
    config : CleaningConfig,
    report : List[Dict],
    removal_counts: Dict[str,int],
    flagged_parts : List[pd.DataFrame]
) -> pd.DataFrame:
    
    step = 'Step 5: Target Cleaning'
    print('\n'+step)
    
    print('-'*len(step))
    
    
    df = df.copy()
    
    target_col = 'elapsed_seconds'
    
    if target_col not in df.columns:
        raise ValueError(f'Target column {target_col} is missing')
     
    df[target_col] = pd.to_numeric(df[target_col],errors='coerce')
    
    missing_target_mask = df[target_col].isna()
    
    add_report(
        report,
        step,
        'missing_target_rows',
        int(missing_target_mask.sum()),
        'Rows with missing elapsed_seconds are removed'
    )
    

    df = remove_rows(
        df= df,
        mask=missing_target_mask,
        reason='Missing elapsed_seconds target',
        removal_key = 'missing_target',
        removal_counts=removal_counts,
        flagged_parts= flagged_parts
    )
    
    
    non_positive_duration_mask = df[target_col] <=0 
    
    add_report(
        report,
        step,
        'non_positive_duration_rows',
        int(non_positive_duration_mask.sum()),
        'Rows with elapsed_seconds <= 0 are removed'
    )
    
    
    df = remove_rows(
        df= df,
        mask = non_positive_duration_mask,
        reason= 'Non-positive elapsed_seconds',
        removal_key = 'invalid_duration',
        removal_counts= removal_counts,
        falgged_parts = flagged_parts
    )
    
    
    
    too_short_duration_mask = df[target_col] < config.min_valid_duration_seconds
    
    
    add_report(
        report,
        step,
        'too_short_duration_rows',
        int(too_short_duration_mask.sum()),
        f'Rows with elapsed_seconds < {config.min_valid_duration_seconds} are removed'
    )
    
    
    df = remove_rows(
        df=  df,
        mask = too_short_duration_mask,
        reason= f'Duration shorter than {config.min_valid_duration_seconds} seconds',
        removal_key= 'invalid_duration',
        removal_counts=removal_counts,
        flagged_parts=flagged_parts
    )
    
    
    q1 = df[target_col].quantile(0.25)
    q3 = df[target_col].quantile(0.75)
    iqr = q3 - q1
    
    iqr_upper = q3 + 3.0* iqr
    
    p99 = df[target_col].quantile(0.99)
    p999 = df[target_col].quantile(0.999)
    
    duration_outlier_mask  = df[target_col] > iqr_upper
    
    
    collect_flagged_rows(
        flagged_parts = flagged_parts,
        df = df,
        mask = duration_outlier_mask,
        reason= f'High duration outlier above IQR upper limit : {iqr_upper:.2f} seconds',
        action= 'flagged_not_removed'
    )
    
    
    
    add_report(
        report,
        step,
        'duration_outlier_flagged_not_removed',
        int(duration_outlier_mask.sum()),
        'High duration outliers are flagged only, not removed , becuse long rides may be valid'
    )
    add_report(report, step, "duration_min", float(df[target_col].min()))
    add_report(report, step, "duration_q1", float(q1))
    add_report(report, step, "duration_median", float(df[target_col].median()))
    add_report(report, step, "duration_q3", float(q3))
    add_report(report, step, "duration_p99", float(p99))
    add_report(report, step, "duration_p999", float(p999))
    add_report(report, step, "duration_max", float(df[target_col].max()))
    add_report(report, step, "duration_iqr_upper_limit", float(iqr_upper))
    
    print("Target cleaning completed.")
    print("Remaining rows:", len(df))
    print("Duration median:", df[target_col].median())
    print("Duration p99:", p99)
    print("Duration max:", df[target_col].max())
    
    return df

In [ ]:
# def check_time_consistency(
#     df : pd.DataFrame,
#     config: CleaningConfig,
#     report: List[Dict],
#     removal_counts : Dict[str,int],
#     flagged_parts : List[pd.DataFrame],
# )-> pd.DataFrame:
#     step = 'Step 6 : Time consitency check'
    
#     print('\n'+step)
#     print('-'*len(step))
    
#     df = df.copy()
    
#     if 'start_time' not in df.columns or 'end_time' not in df.columns:
#         add_report(report,step,'time_consistency_check','skipped','start_time or end_time is missing')
#         return df
    
#     missing_start_mask = df['start_time'].isna()
#     add_report(report,step,'missing_or_invalid_start_time_rows',int(missing_start_mask.sum()))
    
#     df = remove_rows(
#         df,
#         mask = missing_start_mask,
#         reason='Missing or invalid start_time',
#         removal_key='missing_start_time_removed',
#         removal_counts= removal_counts,
#         flagged_parts= flagged_parts
#     )
    
    
#     if 'end_time' not in df.columns:
#         add_report(report,step, 'time_consistency_check','partial','end_time column missing : mismatch validation skipped')
#         return df
    
#     missing_end_mask = df['end_time'].isna()
    
#     collect_flagged_rows(
#         flagged_parts,
#         df,
#         missing_end_mask,
#         reason='Missing or Invalid end_time ; timestamp duration cannot be validate',
#         action = 'flagged_not_removed',
#     )
    
    
#     add_report(report,step,'missing_or_invalid_end_time_rows_flagged_not_removed',int(missing_end_mask.sum()))
    
#     both_available_mask = df['start_time'].notna() & df['end_time'].notna()
    
    
#     invalid_order_mask = both_available_mask & (df['end_time'] <= df['start_time'])
#     add_report(report,step, 'invalid_time_order_rows',int(invalid_order_mask.sum()))
    
#     df = remove_rows(
#         df,
#         invalid_order_mask,
#         reason='Invalid time order:end_time <= start_time',
#         removal_key = 'invalid_time_order_removed',
#         removal_counts= removal_counts,
#         flagged_parts= flagged_parts,
#     )
    
#     both_available_mask = df['start_time'].notna() & df['end_time'].notna()
#     df['_computed_elapsed_secodns'] = np.nan
    
#     df.loc[both_available_mask,'_computed_elapsed_secodns'] = (
#         df.loc[both_available_mask,'end_time'] - df.loc[both_available_mask,'start_time']
#     ).dt.total_seconds()
    
#     tolerance = np.maximum(
#         config.time_mismatch_fixed_tolerance_seconds,
#         config.time_mismatch_relative_tolerance*df[TARGET_COL],
#     )
    
#     mismatch_mask = (
#         both_available_mask
#         & df['_computed_elapsed_secodns'].notna()
#         & ((df['_computed_elapsed_secodns'] - df[TARGET_COL]).abs()> tolerance)
#     )
    
    
#     add_report(
#         report,
#         step,
#         'duration_mismatch_rows',
#         int(mismatch_mask.sum()),
#         f'Tolerance = max({config.time_mismatch_fixed_tolerance_seconds} seconds),',
#         f' {config.time_mismatch_relative_tolerance:.1%} of elapsed_seconds'
#     )
    
#     df = remove_rows(
#         df,
#         mismatch_mask,
#         reason='Large mismatch between timestamp duration and elapsed_seconds',
#         removal_key = 'duration_mismatch_removed',
#         removal_counts=removal_counts,
#         flagged_parts= flagged_parts
#     )
    
#     return df
    
def check_time_consistency(
    df: pd.DataFrame,
    config: CleaningConfig,
    report: List[Dict],
    removal_counts: Dict[str, int],
    flagged_parts: List[pd.DataFrame],
) -> pd.DataFrame:
    
    step = "Step 6: Time Consistency Check"
    
    print("\n" + step)
    print("-" * len(step))
    
    df = df.copy()
    
    if "start_time" not in df.columns or "end_time" not in df.columns:
        add_report(
            report,
            step,
            "time_consistency_check",
            "skipped",
            "start_time or end_time is missing"
        )
        return df
    
    both_available_mask = df["start_time"].notna() & df["end_time"].notna()
    
    add_report(
        report,
        step,
        "rows_with_both_start_and_end_time",
        int(both_available_mask.sum())
    )
    
    invalid_time_order_mask = (
        both_available_mask &
        (df["end_time"] <= df["start_time"])
    )
    
    add_report(
        report,
        step,
        "invalid_time_order_rows",
        int(invalid_time_order_mask.sum()),
        "Rows where end_time is before or equal to start_time are removed."
    )
    
    df = remove_rows(
        df=df,
        mask=invalid_time_order_mask,
        reason="Invalid time order: end_time <= start_time",
        removal_key="invalid_time_order",
        removal_counts=removal_counts,
        flagged_parts=flagged_parts,
    )
    
    both_available_mask = df["start_time"].notna() & df["end_time"].notna()
    
    df["_computed_elapsed_seconds"] = np.nan
    
    df.loc[both_available_mask, "_computed_elapsed_seconds"] = (
        df.loc[both_available_mask, "end_time"] - df.loc[both_available_mask, "start_time"]
    ).dt.total_seconds()
    
    if TARGET_COL in df.columns:
        valid_compare_mask = (
            both_available_mask &
            df["_computed_elapsed_seconds"].notna() &
            df[TARGET_COL].notna()
        )
        
        df["_duration_mismatch_seconds"] = np.nan
        
        df.loc[valid_compare_mask, "_duration_mismatch_seconds"] = (
            df.loc[valid_compare_mask, "_computed_elapsed_seconds"] -
            df.loc[valid_compare_mask, TARGET_COL]
        ).abs()
        
        tolerance = np.maximum(
            300.0,
            0.05 * df.loc[valid_compare_mask, TARGET_COL].astype(float)
        )
        
        mismatch_mask = pd.Series(False, index=df.index)
        
        mismatch_mask.loc[valid_compare_mask] = (
            df.loc[valid_compare_mask, "_duration_mismatch_seconds"] > tolerance
        )
        
        add_report(
            report,
            step,
            "large_duration_mismatch_rows",
            int(mismatch_mask.sum()),
            "Rows removed when timestamp-computed duration differs from elapsed_seconds by more than max(300 seconds, 5%)."
        )
        
        collect_flagged_rows(
            flagged_parts=flagged_parts,
            df= df ,
            mask = mismatch_mask,
            reason="Large mismatch between timestamp duration and elapsed_seconds",
            action='flagged_not_removed'
        )
        add_report(
            report,
            step,
            'large_duration_mismatch_flagged_not_removed',
            int(mismatch_mask.sum()),
            'Large thim mismatch is flagged only , not removed it may be caused by timesstamp format , timezone, or recording diffrences'
            
        )
    
    print("Time consistency check completed.")
    print("Remaining rows:", len(df))
    
    return df

In [10]:
def clean_rider_total(
    df: pd.DataFrame,
    config : CleaningConfig,
    report: List[Dict],
    removal_counts : Dict[str,int],
    flagged_parts : List[pd.DataFrame],
) -> pd.DataFrame:
    
    step ='Step 7 : Rider total cleaning'
    
    if 'rider_total' not in df.columns:
        add_report(report,step,'rider_total_cleaning','skipped','rider_total columns missing')
        return df
    
    print('Rider total descriptive statistics')
    display(df['rider_total'].describe())
    
    missing_mask =df['rider_total'].isna()
    non_positive_mask = df['rider_total'].notna() & (df['rider_total'] <= 0)
    too_large_mask = df['rider_total'].notna() & (df['rider_total']> config.max_reasonable_rider_total)
    
    
    invalid_rider_mask = missing_mask | non_positive_mask | too_large_mask
    
    
    add_report(report,step,'missing_order_total_rows',int(missing_mask.sum()))
    add_report(report,step,'non_positive_rider_total_rows',int(non_positive_mask.sum()))
    add_report(
        report,
        step,
        'too_large_rider_total_rows',
        int(too_large_mask.sum()),
        f' Threshold > {config.max_reasonable_rider_total}. Adjusted if data is bus/shuttle rather than taxi-like rides'
    )
    
    df = remove_rows(
        df,
        invalid_rider_mask,
        reason=f'Invalid rider_total:missing , <= 0 , or> {config.max_reasonable_rider_total}',
        removal_key= 'invalid_rider_total_removed',
        removal_counts= removal_counts,
        flagged_parts=flagged_parts
    )
    return df

In [58]:
def infer_coordinate_system(
    df : pd.DataFrame,
    x_cols : List[str],
    y_cols : List[str],
    min_valid_ratio : float = 0.95
) -> str:
    
    required_cols = list(x_cols) + list(y_cols)
    
    
    if not all(col in df.columns for col in required_cols):
        return 'unknown_missing_columns'
    
    coord_df = df[required_cols].copy()
    
    
    for col in required_cols:
        coord_df[col] = pd.to_numeric(coord_df[col],errors='coerce')
    
    coord_df = coord_df.dropna()
    
    if coord_df.empty:
        return 'unknown_no_valid_coordinates'
    
    x_in_lon_range = coord_df[y_cols].apply(
        lambda s: s.between(-90,90)
    ).all(axis=1)
    
    y_in_lat_range = coord_df[y_cols].apply(
        lambda s: s.between(-90,90)
    ).all(axis=1)
    
    lonlat_ratio = (x_in_lon_range & y_in_lat_range).mean()
    
    if lonlat_ratio >= min_valid_ratio:
        return 'lonlat'
    
    return 'projected_or_unknown'

def clean_coordinates(
    df : pd.DataFrame,
    config : CleaningConfig,
    report : List[Dict],
    removal_counts : Dict[str,int],
    flagged_parts : List[pd.DataFrame],
)-> Tuple[pd.DataFrame,str]:
    
    
    step = 'Step 8 : Coordinate Cleaning'
    
    
    print('\n'+ step)
    print('-'*len(step))
    
    df = df.copy()
    
    
    missing_coord_columns = [col for col in COORD_COLS if col not in df.columns]
    
    if missing_coord_columns:
        add_report(
            report,
            step,
            'coordinate_cleaning',
            'skipped',
            f'Missing coordinate  columns : {missing_coord_columns}'
        )
        
        return df ,'unknown_missing_columns'
    
    
    
    for col in COORD_COLS:
        df[col] = pd.to_numeric(df[col],errors='coerce')
        
        
    missing_coord_mask = df[COORD_COLS].isna().any(axis=1)
    
    
    add_report(
        report,
        step,
        'missing_coordinate_rows',
        int(missing_coord_mask.sum()),
        'Rows with missing start/end coordinate values are removed'
    )
    
    df = remove_rows(
        df= df,
        mask = missing_coord_mask,
        reason = 'Missing coordinate value in start/end x/y columns',
        removal_key = 'invalid_coordinates',
        removal_counts = removal_counts,
        flagged_parts = flagged_parts
    )
    
    coordinate_system = infer_coordinate_system(
        df= df,
        x_cols= X_COLS,
        y_cols= Y_COLS,
    )
    
    add_report(
        report,
        step,
        'inferred_coordinate_system',
        coordinate_system
    )
    
    
    print(f'Inferred coordinate system :{coordinate_system}')
    
    
    if coordinate_system == 'lonlat':
        invalid_lonlate_mask =(
            ~df['start_xcoord'].between(-180,180)|
            ~df['end_xcoord'].between(-180,180)|
            ~df['start_ycoord'].between(-90,90)|
            ~df['end_ycoord'].between(-90,90)
        )
        
        add_report(
            report,
            step,
            'invalid_lonlat_range_rows',
            int(invalid_lonlate_mask.sum()),
            'Rows outside valid longitude/latitude ranges are removed'
        )
        
        df = remove_rows(
                df =df,
                mask = invalid_lonlate_mask,
                reason='Invalid longitude/latitude coordinate range',
                removal_key = 'invalid_coordinate',
                removal_counts=removal_counts,
                flagged_parts= flagged_parts
        )
        
    else:
        add_report(
            report,
            step,
            'lonlat_range_enforcement',
            'skipped',
            'Coordinates appear projected or unknown ; longitude/latitude ranges are not enforced'
        )
        
        
    start_zero_placeholder_mask =(
        df['start_xcoord'].abs() <= config.zero_coordinate_tolerance
    ) & (
        df['start_ycoord'].abs() <= config.zero_coordinate_tolerance
    )
    
    end_zero_placeholder_mask =(
        df['end_xcoord'].abs() <= config.zero_coordinate_tolerance
    ) &(
        df['end_ycoord'].abs() <= config.zero_coordinate_tolerance
    )
    zero_placeholder_mask = start_zero_placeholder_mask | end_zero_placeholder_mask
    
    add_report(
        report,
        step,
        'zero_coordinate_placeholder_rows',
        int(zero_placeholder_mask.sum()),
        'Rows where start or end coordinate pair is effectively (0,0)'
    )
    
    
    df= remove_rows(
        df=df,
        mask = zero_placeholder_mask,
        reason= 'Zero coordinate placeholder in start or end location',
        removal_key='invalid_coordinates',
        removal_counts=removal_counts,
        flagged_parts=flagged_parts
    )
    
    
    constant_coord_cols =[
        col for col in COORD_COLS
        if df[col].nunique(dropna=True) <= 1
    ]
    
    if constant_coord_cols:
        add_report(
            report,
            step,
            'constant_coordinate_columns_detected',
            len(constant_coord_cols),
            ', '.join(constant_coord_cols)
        )
        
        print(
            'Warning: some coordinate columns have one or fewer unique values:',
            constant_coord_cols
        )
        
        
        same_location_mask =(
            (df['start_xcoord'] == df['end_xcoord']) &
            (df['start_ycoord'] == df['end_ycoord'])
        )
        
        
        
        collect_flagged_rows(
            flagged_parts=flagged_parts,
            df=df,
            mask= same_location_mask,
            reason='Start and end coordinates are exactly identical',
            action='flagged_not_removed'
        )
        
        
        add_report(
            report,
            step,
            'same_start_and_coordinate_rows_flagged_not_removed',
            int(same_location_mask.sum()),
            'Same-location trips may be valid short trips , so they are flagged but not automatically removed'
        )
        
        
        print('Coordinate cleaning completed')
        print('Remaning rows: ', len(df))
        
    return df,coordinate_system
        
    

In [12]:
def haversine_distance_km(lon1,lat1,lon2,lat2) -> np.ndarray:
    raduis_km = 6371.0088
    
    lon1 = np.radians(lon1.astype(float))
    lat1 = np.radians(lat1.astype(float))
    lon2 = np.radians(lon2.astype(float))
    lat2 = np.radians(lat2.astype(float))
    
    dlon = lon2 - lon1
    dlat = lat2 - lon1
    
    a = np.sin(dlat/2.0)**2+ np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    a = np.clip(a,0,1)
    c = 2*np.arcsin(np.sqrt(a))
    return raduis_km*c


def check_distance_and_speed(
    df: pd.DataFrame,
    config: CleaningConfig,
    coordinate_system : str,
    report: List[Dict],
    remove_counts : Dict[str,int],
    flagged_parts : List[pd.DataFrame]
) -> pd.DataFrame:
    
    step = 'Step 9: Distance and speed quantity check'
    df = df.copy()
    
    if not all(col in df.columns for col in COORD_COLS):
        add_report(report,step,'distance_speed_check','skipped','coordinate columns missing')
        return df
    
    if TARGET_COL not in df.columns:
        add_report(report,step,'distance_speed_check','skipped','target column missing')
        return df
    
    if coordinate_system == 'lonlat':
        df['_validation_distance_km'] = haversine_distance_km(
            df['start_xcoord'],df['start_ycoord'],df['end_xcoord'],df['end_ycoord']
        )
        
        elapsed_hours = df[TARGET_COL]/3600.0
        df['_validation_speed_kmh'] = np.where(elapsed_hours>0 , df['_validation_distance_km']/elapsed_hours,np.nan)
        
        
        print('Validation distance statistics,km:')
        display(df['_validation_distance_km'].describe())
        print('\nValidation speed statitcs , km/h:')
        display(df['_validation_distance_km'].describe())
        
        
        
        unrealistic_speed_mask = (
            df['_validation_speed_kmh'].notna()
            & (df['_validation_distance_km'] >= config.min_distance_for_speed_check_km)
            & (df['_validation_speed_kmh']> config.max_reasonable_speed_kmh)
        )
        
        unrealistic_distance_mask = (
            df['_validation_distance_km'].notna()
            & (df['_validation_distance_km']> config.max_reasonable_trip_distance_km)
        )
        
        unrealistic_mask = unrealistic_speed_mask | unrealistic_distance_mask
        
        add_report(
            report,
            step,
            'unrealistic_speed_rows',
            int(unrealistic_speed_mask.sum()),
            f'speed > {config.max_reasonable_speed_kmh} km/h'
        )
        add_report(
            report,
            step,
            'unrealistic_distance_rows',
            int(unrealistic_distance_mask.sum()),
            f'distance > {config.max_reasonable_trip_distance_km} km'
        )
        
        # df = remove_rows(
        #     df,
        #     unrealistic_mask,
        #     reason='Unrealistic validation speed or trip distance from lon/lat coordinates',
        #     removal_key=  'unrealistic_speed_or_distance_removed',
        #     removal_counts= remove_counts,
        #     flagged_parts= flagged_parts,
        # )
        
        collect_flagged_rows(
            flagged_parts=flagged_parts,
            df= df,
            mask = unrealistic_speed_mask,
            reason= 'Suspicious or unrealistic speed',
            action= 'flagged_not_removed'
        )
        
        add_report(
            report,
            step,
            'unrealistic_speed_flagged_not_removed',
            int(unrealistic_distance_mask.sum()),
            'Speed outliers are flagged only , not removed Speed depends on coordinate system and distance unit'
        )
    else :
        df['_validation_euclidean_distance_raw_units'] =np.sqrt(
            (df['end_xcoord']-df['start_xcoord']) **2 
            + (df['end_ycoord']-df['start_ycoord'])**2
        )
        
        
        print('Projected/unknown coordinate system detected')
        print('Euclidean distance is calculated only as a rough validation signal; no speed-based removal is applied')
        display(df['_validation_euclidean_distance_raw_units'].describe())
        
        
        if len(df) >= config.min_rows_for_duration_outlier_rule:
            threshold = df['_validation_euclidean_distance_raw_units'].quantile(config.projected_distance_upper_percentile)
            suspicious_distance_mask = df['_validation_euclidean_distance_raw_units']>threshold
            
            collect_flagged_rows(
                flagged_parts,
                df,
                suspicious_distance_mask,
                reason=f'Extreme projected/unknown Euclidean distance above  q{config.projected_distance_upper_percentile:.3f}',
                action='flagged_not_removed'
            )
            
            add_report(
                report,
                step,
                'projected_distance_outliers_flagged_not_removed',
                int(suspicious_distance_mask.sum()),
                f'Threshold = {threshold:.4f} raw coordinate units. Not because units are   unknown'
            )
            
        else:
            add_report(
                report,
                step,
                'projected_distance_outlier_rule',
                'skipped',
                f'Only {len(df)} rows available; threshold requires at least {config.min_rows_for_duration_outlier_rule} rows'
            )
            
    return df

In [13]:
def clean_carrier_code(df:pd.DataFrame,config:CleaningConfig,report:List[Dict]) -> pd.DataFrame:
    step = 'Step 10: Categorical column cleaning'
    df = df.copy()
    
    
    if 'carrier_code' not in df.columns:
        add_report(report,step,'carrier_code_cleaning','skipped', 'carrier_code column missing')
        return df
    
    df['carrier_code'] = df['carrier_code'].astype('string').str.strip().str.upper()
    df.loc[df['carrier_code'].isin(['','<NA>','NAN','NONE','NULL']),'carrier_code'] = pd.NA
    
    missing_count = int(df['carrier_code'].isna().sum())
    value_counts = df['carrier_code'].value_counts(dropna=True)
    rare_categories = df['carrier_code'].value_counts(dropna=True)
    rare_categories = rare_categories[rare_categories<config.rare_category_min_count]
    
    print('carrier_code value counts:')
    display(value_counts.head(50))
    
    
    add_report(report,step,'carrier_code_missing_rows',missing_count)
    add_report(report,step,'carrier_code_unique_non_missing',int(df['carrier_code'].nunique(dropna=True)))
    
    add_report(
        report,
        step,
        'carrier_code_rare_categories',
        int(len(rare_categories)),
        f'Categories with count < {config.rare_category_min_count}:{rare_categories.index.tolist()[:30]}'
    )
    
    return df


def clean_save_forward_marker(df:pd.DataFrame,report:List[Dict])-> pd.DataFrame:
    step = 'Step 10: Categorical column cleaning'
    df = df.copy()
    
    if 'save_forward_marker' not in df.columns:
        add_report(report,step,'save_forward_marker_cleaning','skipped','save_forward_marker column missing')
        return df
    
    original_unique = df['save_forward_marker'].dropna().unique().tolist()
    
    yes_values = {'y','yes','true','t','1'}
    no_values = {'n','no','false','f','0'}
    missing_values = {'','<na>','nan','none','null'}
    def standardize_marker(value):
        if pd.isna(value):
            return pd.NA
    
        text = str(value).strip().lower()
    
        if text in missing_values:
            return pd.NA

        if text in yes_values:
            return 'yes'
        
        if text in no_values:
            return 'no'
        
        return text
    
    df['save_forward_marker'] = df['save_forward_marker'].apply(standardize_marker).astype('string')
    
    print('save_forward_marker original unique values:')
    print(original_unique[:50])
    
    
    
    
    print('\nsave_forward_marker cleaned value counts:')
    
    display(df['save_forward_marker'].value_counts(dropna=False))
    
    add_report(
        report,
        step,
        'save_forward_marker_missing_rows',
        int(df['save_forward_marker'].isna().sum())
    ) 
    
    
    add_report(
        report,
        step,
        'save_forward_marker_unique_non_missing',
        int(df['save_forward_marker'].nunique(dropna=True))
    )
    
    return df
def clean_categorical_columns(df:pd.DataFrame,config:CleaningConfig,report:List[Dict])-> pd.DataFrame:
    
    df = clean_carrier_code(df,config,report)
    df = clean_save_forward_marker(df,report)
    
    return df

In [14]:
def build_summary_report(
    df: pd.DataFrame,
    original_rows : int,
    removal_counts: Dict[str,int],
    report:List[Dict]
)-> List[str]:
    
    cleaned_df = remove_helper_columns(df)
    final_rows = len(cleaned_df)
    removal_rows = original_rows - final_rows
    removed_pct = safe_percent(removal_rows,original_rows)
    
    summary_lines = []
    summary_lines.append("="*70)
    summary_lines.append('Phase 1 Data Cleaning')
    summary_lines.append('='*70)
    summary_lines.append(f'Original rows:{original_rows:,}')
    summary_lines.append(f'Final removed rows: {removal_rows:,}({removed_pct:.2f}%)')  
    summary_lines.append('')
    summary_lines.append('Rows removed by major reason:')
    
    for key in REMOVAL_COUNT_KEYS:
        summary_lines.append(f'-{key}:{int(removal_counts.get(key,0)):,}')
        add_report(report,'Step 11: Cleaning summery report',key,int(removal_counts.get(key,0)))
        
    remaining_missing = cleaned_df.isna().sum().sort_values(ascending=False)
    remaining_missing = remaining_missing[remaining_missing > 0]
    
    summary_lines.append('')
    summary_lines.append('Remaining missing values:')
    if len(remaining_missing) == 0:
        summary_lines.append('-None')
    else:
        for col,count in remaining_missing.items():
            summary_lines.append(f'-{col}:{int(count):,}')
            
    summary_lines.append('')
    summary_lines.append('Final data Types:')
    for col ,dtype in cleaned_df.dtypes.items():
        summary_lines.append(f'- {col}:{dtype}')
        
    summary_lines.append('')
    summary_lines.append(f'Final shape: {cleaned_df.shape}')
    summary_lines.append('='*70)
    
    print('\n'.join(summary_lines))
    
    add_report(report , 'Step 11 : Cleaning summary report ,report','original_rows',original_rows)
    add_report(report,'Step 11 : Cleaning summary report ,report', 'final_rows',final_rows)
    add_report(report,'Step 11: Cleaning summary report','total_removed_rows',removal_rows,f'{removed_pct:.2f}%')
    add_report(report,'Step 11 : Cleaning summary report','final_shape',str(cleaned_df.shape))
    
    return summary_lines

In [15]:

def save_output_files(
    df: pd.DataFrame,
    report: List[Dict],
    summary_lines:list[str],
    flagged_parts : List[pd.DataFrame],
    config: CleaningConfig,
) -> Tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    
    cleaned_df = remove_helper_columns(df)
    cleaned_df.to_csv(config.cleaned_output_file,index=False)
    
    report_df = pd.DataFrame(report)
    report_df.to_csv(config.report_csv_file,index=False)
    
    
    with open(config.report_txt_file,'w',encoding='utf-8') as f:
        f.write('\n'.join(summary_lines))
        f.write('\n\n')
        f.write('Detailes structured report is available in cleaning_report.csv \n')
        
        
    if flagged_parts:
        flagged_df = pd.concat(flagged_parts,ignore_index=True,sort=False)
    else:
        flagged_df = pd.DataFrame(columns=list(cleaned_df.columns)+ ['flag_reason','flag_action'])
        
    flagged_df.to_csv(config.flagged_output_file,index=False)
    
    
    print('Saved output files:')
    print(f'- cleaned dataset:{config.cleaned_output_file}')
    print(f'-Cleaning report CSV : {config.report_csv_file}')
    print(f'-Flagged rows/outliers: {config.flagged_output_file}')
    
    return cleaned_df,report_df,flagged_df


In [16]:
def run_cleaning_pipeline(config: CleaningConfig):
    """Run the full Phase 1 data cleaning pipeline from top to bottom."""
    report: List[Dict] = []
    flagged_parts: List[pd.DataFrame] = []
    removal_counts = initialize_removal_counts()

    print("Starting Phase 1 data cleaning pipeline...")

    df, original_rows = load_and_inspect_data(config, report)

    analyze_missing_values(df, config, report)

    df = remove_duplicates(df, report, removal_counts, flagged_parts)

    df = convert_data_types(df, report, flagged_parts)

    df = clean_target(df, config, report, removal_counts, flagged_parts)

    df = check_time_consistency(df, config, report, removal_counts, flagged_parts)

    df = clean_rider_total(df, config, report, removal_counts, flagged_parts)

    df, coordinate_system = clean_coordinates(df, config, report, removal_counts, flagged_parts)

    df = check_distance_and_speed(df, config, coordinate_system, report, removal_counts, flagged_parts)

    df = clean_categorical_columns(df, config, report)

    summary_lines = build_summary_report(df, original_rows, removal_counts, report)

    cleaned_df, report_df, flagged_df = save_output_files(df, report, summary_lines, flagged_parts, config)

    print("\nPipeline finished successfully.")
    return cleaned_df, report_df, flagged_df

In [17]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nFiles in current directory:")
for p in Path.cwd().iterdir():
    print(p.name)

print("\nDoes student_version.csv exist?")
print(Path("student_version.csv").exists())

Current working directory:
c:\Users\mojta_tok6stl\OneDrive\Desktop\learning_ai\University-Project

Files in current directory:
categorical_feature_mapping.json
cleaned_student_version.csv
cleaning_report.csv
cleaning_report.txt
document.pdf
feature_columns.txt
feature_engineered_student_version.csv
feature_engineering_report.csv
feature_engineering_report.txt
flagged_outliers.csv
project.ipynb
student_version.csv

Does student_version.csv exist?
True


In [18]:
INPUT_FILE = "student_version.csv"

config = CleaningConfig(
    input_file=INPUT_FILE,
    cleaned_output_file="cleaned_student_version.csv",
    report_csv_file="cleaning_report.csv",
    report_txt_file="cleaning_report.txt",
    flagged_output_file="flagged_outliers.csv",
    max_reasonable_rider_total=20.0,
    max_reasonable_speed_kmh=160.0,
    min_valid_duration_seconds=10.0,
)

run_cleaning_pipeline(config)

Starting Phase 1 data cleaning pipeline...
Loading dataset...
Raw Shape: (1048575, 11)
Raw Columns:
['record_key', 'carrier_code', 'start_time', 'end_time', 'rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'save_forward_marker', 'elapsed_seconds']

RAw dtypes: 


record_key              object
carrier_code             int64
start_time              object
end_time                object
rider_total              int64
start_xcoord           float64
start_ycoord           float64
end_xcoord             float64
end_ycoord             float64
save_forward_marker     object
elapsed_seconds          int64
dtype: object


 First rows before normalization


,record_key,carrier_code,start_time,end_time,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,save_forward_marker,elapsed_seconds
0,rid2875421,2,3/14/2016 17:24,3/14/2016 17:32,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,rid2377394,1,6/12/2016 0:43,6/12/2016 0:54,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,rid3858529,2,1/19/2016 11:35,1/19/2016 12:10,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,rid3504673,2,4/6/2016 19:32,4/6/2016 19:39,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,rid2181028,2,3/26/2016 13:30,3/26/2016 13:38,1,-73.973053,40.793209,-73.972923,40.782520,N,435



 Column normalization mapping:


...

carrier_code
2    559171
1    485908
Name: count, dtype: Int64

save_forward_marker original unique values:
['N', 'Y']

save_forward_marker cleaned value counts:


save_forward_marker
no     1039339
yes       5740
Name: count, dtype: Int64

Phase 1 Data Cleaning
Original rows:1,048,575
Final removed rows: 3,496(0.33%)

Rows removed by major reason:
-missing_target_removed:0
-exact_duplicate_rows_removed:0
-duplicate_record_keys_removed:0
-invalid_duration_removed:1,412
-duration_outlier_removed:0
-missing_start_time_removed:0
-invalid_time_order_removed:0
-duration_mismatch_removed:0
-invalid_rider_total_removed:19
-invalid_coordinates_removed:0
-unrealistic_speed_or_distance_removed:0

Remaining missing values:
-None

Final data Types:
- record_key:object
- carrier_code:string
- start_time:datetime64[ns]
- end_time:datetime64[ns]
- rider_total:int64
- start_xcoord:float64
- start_ycoord:float64
- end_xcoord:float64
- end_ycoord:float64
- save_forward_marker:string
- elapsed_seconds:int64

Final shape: (1045079, 11)
Saved output files:
- cleaned dataset:cleaned_student_version.csv
-Cleaning report CSV : cleaning_report.csv
-Flagged rows/outliers: flagged_outliers.csv

Pipeline finished successfully.


(         record_key carrier_code          start_time            end_time  rider_total  start_xcoord  start_ycoord  \
 0        rid2875421            2 2016-03-14 17:24:00 2016-03-14 17:32:00            1    -73.982155     40.767937   
 1        rid2377394            1 2016-06-12 00:43:00 2016-06-12 00:54:00            1    -73.980415     40.738564   
 2        rid3858529            2 2016-01-19 11:35:00 2016-01-19 12:10:00            1    -73.979027     40.763939   
 3        rid3504673            2 2016-04-06 19:32:00 2016-04-06 19:39:00            1    -74.010040     40.719971   
 4        rid2181028            2 2016-03-26 13:30:00 2016-03-26 13:38:00            1    -73.973053     40.793209   
 ...             ...          ...                 ...                 ...          ...           ...           ...   
 1048570  rid0002921            1 2016-04-06 14:16:00 2016-04-06 14:20:00            1    -73.973015     40.760948   
 1048571  rid1329189            2 2016-03-24 01:26:00 20

In [19]:
from pathlib import Path

expected_outputs = [
    "cleaned_student_version.csv",
    "cleaning_report.csv",
    "cleaning_report.txt",
    "flagged_outliers.csv",
]

for file in expected_outputs:
    path = Path(file)
    print(file, "=>", "CREATED" if path.exists() else "NOT FOUND")

cleaned_student_version.csv => CREATED
cleaning_report.csv => CREATED
cleaning_report.txt => CREATED
flagged_outliers.csv => CREATED


## Phase 2: Feature Engineering


In [20]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict,List,Tuple
import json

import numpy as np
import pandas as pd

In [21]:
@dataclass

class FeatureEngineeringConfig:
    input_file : str = 'cleaned_student_version.csv'
    engineered_output_file : str = 'feature_engineered_student_version.csv'
    report_csv_file : str = 'feature_engineering_report.csv'
    report_txt_file : str = 'feature_engineering_report.txt'
    feature_columns_file : str =  'feature_columns.txt'
    categorical_mapping_file: str = 'categorical_feature_mapping.json'
    
    target_col : str = 'elapsed_seconds'
    record_key_col : str = 'record_key'
    
    start_time_col : str = 'start_time'
    end_time_col : str = 'end_time'
    
    start_x_col : str = 'start_xcoord'
    start_y_col : str =  'start_ycoord'
    end_x_col : str  = 'end_xcoord'
    end_y_col : str = 'end_ycoord'
    
    
    
    carier_col : str = 'carrier_code'
    marker_col : str = 'save_forward_marker'
    
    
    rare_category_min_count : int = 10
    
    morning_rush_start : int = 7
    morning_rush_end : int = 10
    evening_rush_start : int = 16
    evening_rush_end :  int = 19
    
    
    create_cyclical_time_features : bool = True
    
    one_hot_encode_categoricals : bool = True
    
    include_raw_coordinates_as_features: bool = True
    
    include_record_key_in_output : bool = True


EXPECTED_COLUMNS = [
    "record_key",
    "carrier_code",
    "start_time",
    "end_time",
    "elapsed_seconds",
    "rider_total",
    "start_xcoord",
    "start_ycoord",
    "end_xcoord",
    "end_ycoord",
    "save_forward_marker",
]

In [22]:
def add_report(report:List[Dict],step:str,metric:str,value,details:str='')-> None:
    report.append({
        'step': step,
        'metric' : metric,
        'value':value,
        'details': details
    })
    
def safe_read_csv(path:str)-> pd.DataFrame:
    file_path = Path(path)
    
    if not file_path.exists():
        raise FileNotFoundError(
            f'Input file not found : {file_path.resolve()}\n'
            'Phase 2 expects the cleaned dataset from Phase 1'
            'Please run phase 1 first and make sure cleaned_student_version.csv exists'
        )
    return pd.read_csv(file_path)

def display_missing_summery(df:pd.DataFrame)-> pd.DataFrame:
    summery = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_percent': df.isna().mean()*100
    }).sort_values('missing_percent',ascending=False)
    display(summery)
    return summery



def infer_coordinate_system(
    df: pd.DataFrame,
    x_cols : List[str],
    y_cols : List[str],
    min_valid_ratio: float = 0.95
)-> str:
    
    existing_x = [c for c in x_cols if c in df.columns]
    existing_y = [c for c in y_cols if c in df.columns]
    
    
    if not existing_x or not existing_y:
        return 'missing_coordinate'
    
    
    checks = []
    
    for col in existing_x:
        s = pd.to_numeric(df[col],errors='coerce').dropna()
        checks.append(len(s)>0 and float(((s>= -180)&(s<= 180)).mean())>= min_valid_ratio)
        
        
    for col in existing_y:
        s = pd.to_numeric(df[col],errors='coerce').dropna()
        checks.append(len(s)>0 and float(((s>= -90) & (s<= 90)).mean())>= min_valid_ratio)
        
        
    if all(checks):
        return 'longitude_latitude'
    
    
    return 'projected_or_unknown'


def haversine_distance_km(lon1:pd.Series,lat1:pd.Series,lon2 :pd.Series,lat2:pd.Series) -> pd.Series:
    radius_km =  6371.0088
    
    lon1_rad = np.radians(lon1.astype(float))
    lat1_rad = np.radians(lat1.astype(float))
    lon2_rad = np.radians(lon2.astype(float))
    lat2_rad = np.radians(lat2.astype(float))
    
    
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    
    a = (
        np.sin(dlat/2.0)**2+
        np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(dlon/2.0)**2
    )
    
    
    c = 2*np.arcsin(np.sqrt(a))
    return radius_km*c



def euclidean_distance(x1:pd.Series,y1:pd.Series,x2 : pd.Series,y2:pd.Series) -> pd.Series:
    
    
    dx = x2.astype(float) - x1.astype(float)
    dy = y2.astype(float) - y1.astype(float)
    return np.sqrt(dx**2+dy**2)


def standarize_save_forward_marker(value):
    if pd.isna(value):
        return 'UNKNOWN'
    
    text = str(value).strip().lower()
    
    yes_values = {'y','yes','true','t','1'}
    no_values = {'n','no','false','f','0'}
    missing_valuse = {'',"<na>",'nan','none','null'}
    
    if text in missing_valuse:
        return 'UNKNOWN'
    if text in yes_values:
        return 'YES'
    if text in no_values:
        return 'NO'
    
    return text.upper()


def group_rare_categories(series:pd.Series,min_count:int)-> Tuple[pd.Series,Dict[str,int]]:
    
    
    s = series.astype('string').str.strip().str.upper()
    s = s.replace({"":pd.NA, '<NA>': pd.NA,"NONE":pd.NA,'NULL':pd.NA})
    s = s.fillna('UNKNOWN')
    
    counts =  s.value_counts(dropna=False).to_dict()
    rare_values = [cat for cat,count in counts.items() if count < min_count]
    
    s = s.where(~s.isin(rare_values),'RARE')
    return s , counts

def remove_duplicate_columns(df:pd.DataFrame)-> pd.DataFrame:
    return df.loc[:,~df.columns.duplicated()].copy()


In [23]:
def load_and_inspect_cleaned_data(cofing: FeatureEngineeringConfig,report:List[Dict]) -> pd.DataFrame:
    
    step = 'Step 1 : Load and Inspect cleaned DAta'
    df = safe_read_csv(cofing.input_file)
    
    print("Loaded Cleaned DataSet:")
    print('Shape:',df.shape)
    
    if df.empty:
        raise ValueError(
            f'{config.input_file} is empty'
            "Phase 2  cannot run on an empty cleaned dataset",
            'Go back to Phase 1 and check which cleaning rule removed all rows'
        )
    
    
    print('\nColumns:')
    display(pd.DataFrame({'Columns':df.columns}))
    
    print('\nData Types')
    display(df.dtypes)
    
    print('\n First rows')
    display(df.head())
    
    
    missing_expected = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    
    add_report(report,step,'input_file',config.input_file)
    add_report(report,step,'input_rows',len(df))
    add_report(report,step,'input_columns',df.shape[1])
    add_report(report,step,'missing_expected_columns',len(missing_expected),str(missing_expected))
    
    if config.target_col not in df.columns:
        raise ValueError(
            f"Target column :'{config.target_col}' is missing"
            'Phase 2 requires elapsed_seconds from phase 1'
            )
        
    return df

In [24]:
import pandas as pd
from pathlib import Path

for file in [
    "student_version.csv",
    "cleaned_student_version.csv",
    "feature_engineered_student_version.csv",
]:
    if Path(file).exists():
        df_tmp = pd.read_csv(file)
        print(file, "=>", df_tmp.shape)
    else:
        print(file, "=> NOT FOUND")

student_version.csv => (1048575, 11)
cleaned_student_version.csv => (1045079, 11)
feature_engineered_student_version.csv => (1045079, 25)


In [25]:
def validate_cleaned_data(df:pd.DataFrame,config:FeatureEngineeringConfig,report:List[Dict])-> pd.DataFrame:
    step = 'Step 2: Basic Validation befor featur engineering'
    
    df = df.copy()
    
    print('\nMissing value summery')
    display_missing_summery(df)
    
    
    add_report(report,step,'rows_before_feature_engineering',len(df))
    add_report(report,step, 'columns_before_feature_engineering',df.shape[1])
    add_report(report,step,'target_missing_rows',int(df[config.target_col].isna().sum()))
    
    before = len(df)
    df = df.loc[df[config.target_col].notna()].copy()
    removed =  before - len(df)
    
    add_report(
        report,
        step,
        'rows_removed_missing_target_in_phase2',
        removed,
        "This should usally be 0 if Phase 1 was completed correctly"
    )
    
    return df

In [26]:
def add_time_features(
    df: pd.DataFrame,
    config: FeatureEngineeringConfig,
    report : List[Dict],
)-> Tuple[pd.DataFrame,List[str]]:
    step = 'Step 3: Time-based feature engineering'
    
    
    df = df.copy()
    created_features : List[str] = []
    
    
    if config.start_time_col not in df.columns:
        add_report(report,step,'time_features','skipped','start_time column missing')
        return df,created_features
    
    
    df[config.start_time_col] = pd.to_datetime(df[config.start_time_col],errors='coerce')
    invalid_start_time = int(df[config.start_time_col].isna().sum())
    add_report(report,step,'invalid_or_missing_start_time_rows',invalid_start_time)
    
    df['start_hour'] = df[config.start_time_col].dt.hour
    df['start_day_of_week'] = df[config.start_time_col].dt.dayofweek
    df['start_month'] = df[config.start_time_col].dt.month
    df['is_weekend'] = df['start_day_of_week'].isin([5,6]).astype('Int64')
    
    
    morning_rush = (df['start_hour']>= config.morning_rush_start) & (df['start_hour'] < config.morning_rush_end)
    evening_rush = (df['start_hour'] >= config.evening_rush_start) & (df['start_hour']< config.evening_rush_end)
    
    df['is_rush_hour'] = (morning_rush | evening_rush).astype('Int64')
    
    
    created_features.extend([
        'start_hour',
        'start_day_of_week',
        'start_month',
        'is_weekend',
        'is_rush_hour'
    ])
    
    
    
    if config.create_cyclical_time_features:
        df['start_hour_sin'] = np.sin(2*np.pi*df['start_hour']/24)
        df['start_hour_cos'] = np.cos(2*np.pi*df['start_hour']/24)
        df['start_day_of_week_sin'] = np.sin(2*np.pi*df['start_day_of_week']/7)
        df['start_day_of_week_cos'] = np.cos(2*np.pi*df['start_day_of_week']/7)
        df['start_month_sin'] = np.sin(2*np.pi*df['start_month']/12)
        df['start_month_cos'] = np.cos(2*np.pi*df['start_month']/12)
        
        
        created_features.extend([
            'start_hour_sin',
            'start_hour_cos',
            'start_day_of_week_sin',
            'start_day_of_week_cos',
            'start_month_sin',
            'start_month_cos'
        ])
        
    print('Created time features:')
    print(created_features)
    
    display(df[created_features].head())
    
    
    add_report(report,step,'created_time_features',len(created_features),str(created_features))
    add_report(
        report,
        step,
        'rush_hour_rule',
        'created',
        f'Morning: [{config.morning_rush_start},{config.morning_rush_end}]'
        f'Evening: [{config.evening_rush_start}, {config.evening_rush_end}]'
    )
    
    return df,created_features

In [59]:
def add_distance_features(
    df : pd.DataFrame,
    config: FeatureEngineeringConfig,
    report : List[Dict],
)-> Tuple[pd.DataFrame,List[str],str]:
    
    
    step = 'Step 4: Distance and spatial feature engineering'
    
    df = df.copy()
    
    created_features : List[str] = []
    
    coord_cols = [
        config.start_x_col,
        config.start_y_col,
        config.end_x_col,
        config.end_y_col
    ]
    
    missing_coord_cols = [col for col in coord_cols if col not in df.columns]
    
    
    if missing_coord_cols:
        add_report(
            report,
            step,
            'distance_features',
            'skipped',
            f'Missing coordinate columns : {missing_coord_cols}'
        )
        
        return df,created_features,'missing_coordinates'
    
    
    for col in coord_cols:
        df[col] = pd.to_numeric(df[col],errors='coerce')
        
    coordinate_system=  infer_coordinate_system(
        df= df,
        x_cols= [config.start_x_col,config.end_y_col],
        y_cols= [config.start_y_col,config.end_y_col]
    )
    
    
    add_report(report,step,'coordinate_system_inferred',coordinate_system)
    
    valid_coord_mask = df[coord_cols].notna().all(axis=1)
    
    add_report(
        report,
        step,
        'rows_with_complete_coordinates',
        int(valid_coord_mask.sum())
    )
    
    df['x_delta'] = df[config.end_x_col] - df[config.start_x_col]
    df['y_delta'] = df[config.end_y_col] - df[config.start_y_col]
    df['abs_x_delta'] = df['x_delta'].abs()
    df['abs_y_delta'] = df['y_delta'].abs()
    
    
    df['same_start_end_location'] = (
        (df[config.start_time_col]== df[config.end_x_col]) &
        (df[config.start_y_col] == df[config.end_y_col])
    ).astype('Int64')
    
    
    
    created_features.extend([
        'x_delta',
        'y_delta',
        'abs_x_delta',
        'abs_y_delta',
        'sama_start_end_location'
    ])
    
    
    if coordinate_system == 'lonlat':
        df['trip_distance_km'] = np.nan
        
        
        df.loc[valid_coord_mask,'trip_distance_km'] = haversine_distance_km(
            lon1 =df.loc[valid_coord_mask,config.start_x_col],
            lat1 = df.loc[valid_coord_mask,config.start_y_col],
            lon2 = df.loc[valid_coord_mask,config.end_x_col],
            lat2 = df.loc[valid_coord_mask,config.end_y_col]
        )
        
        df['trip_distance'] = df['trip_distance_km']
        df['trip_distance_unit'] = 'km'
        
        
        created_features.extend([
            'trip_distance_km',
            'trip_distance'
        ])
    else:
        df['trip_distance_projected'] = np.nan
        
        
        df.loc[valid_coord_mask,'trip_distance_projected'] = euclidean_distance(
            
            x1 = df.loc[valid_coord_mask,config.start_x_col],
            y1 = df.loc[valid_coord_mask,config.start_y_col],
            x2 = df.loc[valid_coord_mask,config.end_x_col],
            y2 = df.loc[valid_coord_mask,config.end_y_col]
        )
        
        df['trip_distance'] = df['trip_distance_projected']
        df['trip_distance_unit'] = 'projected_or_unknown_unit'
        
        
        created_features.extend([
            'trip_distance_projected',
            'trip_distance'
        ])
        
        
        df['trip_distance_log1p'] = np.log1p(df['trip_distance'].clip(lower=0))
        
        created_features.append('trip_distance_log1p')
        
        print('Coordinate system inferred: ', coordinate_system)
        print('Created spatial/distance features: ')
        print(created_features)
        
        
        
        display(df[[c for c in created_features if c in df.columns]].head())
        
        
        add_report(
            report,
            step,
            'created_distance_features',
            len(created_features),
            str(created_features)
        )
        
        
        if 'trip_distance' in df.columns and df['trip_distance'].notna().any():
            add_report(report,step,'trip_distance_missing_rows',int(df['trip_distance'].isna().sum()))
            add_report(report,step,'trip_distance_min',float(df['trip_distance'].min(skipna=True)))
            add_report(report,step,'trip_distance_max',float(df['trip_distance'].max(skipna=True)))
            add_report(report,step,'trip_distance_median',float(df['trip_distance'].median(skipna=True)))
            
        else:
            add_report(
                report,
                step,
                'trip_distance_status',
                'not_created_or_all_missing',
                'No valid trip_distance values were created'
            )
            
        return df,created_features,coordinate_system

In [ ]:

def add_categorical_features(
    df : pd.DataFrame,
    config: FeatureEngineeringConfig,
    report: List[Dict],
) -> Tuple[pd.DataFrame,List[str],Dict]:
    step = 'Step 5:  Categorical feature preparation'
    
    df = df.copy()
    
    created_features : List[str] = []
    mapping_info : Dict = {}
    categorical_base_cols : List[str] = []
    
    
    if config.carier_col in df.columns:
        df['carrier_code_clean'], carrier_counts = group_rare_categories(
            df[config.carier_col],
            min_count=config.rare_category_min_count
        )
        
        categorical_base_cols.append('carrier_code_clean')
        mapping_info['carrier_code_original_counts'] = {str(k) : int(v) for k , v in carrier_counts.items()}
        
        
        add_report(report,step,'carrier_code_unique_after_grouping',int(df['carrier_code_clean'].nunique(dropna=True)))
        
        print('carrier_code_clean value counts:')
        display(df['carrier_code_clean'].value_counts(dropna=False).head(50))
    else:
        add_report(report,step,'carrier_code','skipped','carrier_code column missing')
        
    if config.marker_col in df.columns:
        df['save_forward_marker_clean'] = df[config.carier_col].apply(standarize_save_forward_marker).astype('string')
        
        categorical_base_cols.append('save_forward_marker_clean')
        
        add_report(report,step,'save_forward_marker_after_cleaning',int(df['save_forward_marker_clean'].nunique(dropna=True)))
        print('save_forward_marker_clean value counts:' )
        display(df['save_forward_marker_clean'].value_counts(dropna=False))
        
    else:
        add_report(report,step,'save_forward_marker','skipped','save_forward_marker column missing')
        
    if not config.one_hot_encode_categoricals:
        created_features.extend(categorical_base_cols)
        return df,created_features,mapping_info
    
    if categorical_base_cols:
        encoded_df = pd.get_dummies(
            df[categorical_base_cols],
            columns=categorical_base_cols,
            prefix=categorical_base_cols,
            dummy_na=False,
            dtype=int
        )
        
        encoded_df = remove_duplicate_columns(encoded_df)
        df = pd.concat([df,encoded_df],axis=1)
        
        
        encoded_cols = list(encoded_df.columns)
        created_features.extend(encoded_df)
        mapping_info['one_hot_encoded_columns'] = list(encoded_df.columns)
        
        
        add_report(report,step,'one_hot_encoded_features',len(encoded_df),str(encoded_cols[:50])) 
        print('Created',step,'one_hot_encoded_featurs:',len(encoded_cols),str(encoded_cols[:50]))
        print(encoded_cols)
        
        
    return df,created_features,mapping_info

In [ ]:
def build_feature_dataset(
    df: pd.DataFrame,
    config: FeatureEngineeringConfig,
    time_features: List[str],
    distance_features: List[str],
    categorical_features: List[str],
    report: List[Dict],
) -> Tuple[pd.DataFrame, List[str]]:
    
    step = "Step 6: Build final feature dataset"
    
    df = df.copy()
    

    forbidden_cols = {
        config.target_col,
        config.record_key_col,
        config.start_time_col,
        config.end_time_col,
        "trip_distance_unit",
    }
    
    feature_columns: List[str] = []
    
    raw_coordinate_features: List[str] = []
    
    if config.include_raw_coordinates_as_features:
        for col in [
            config.start_x_col,
            config.start_y_col,
            config.end_x_col,
            config.end_y_col,
        ]:
            if col in df.columns:
                raw_coordinate_features.append(col)
    
    numeric_base_features: List[str] = []
    
    if "rider_total" in df.columns:
        numeric_base_features.append("rider_total")
    
    feature_columns.extend(numeric_base_features)
    feature_columns.extend(raw_coordinate_features)
    feature_columns.extend(time_features)
    feature_columns.extend(distance_features)
    feature_columns.extend(categorical_features)
    
    feature_columns = list(dict.fromkeys([
        col for col in feature_columns
        if col in df.columns
    ]))
    
    feature_columns = [
        col for col in feature_columns
        if col not in forbidden_cols
    ]
    
    numeric_feature_columns: List[str] = []
    removed_non_numeric_features: List[str] = []
    
    for col in feature_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_feature_columns.append(col)
        else:
            removed_non_numeric_features.append(col)
    
    feature_columns = numeric_feature_columns
    
    if removed_non_numeric_features:
        add_report(
            report,
            step,
            "non_numeric_features_removed",
            len(removed_non_numeric_features),
            str(removed_non_numeric_features)
        )
        
        print("Non-numeric features removed from model feature list:")
        print(removed_non_numeric_features)
    
    output_columns: List[str] = []
    
    if config.include_record_key_in_output and config.record_key_col in df.columns:
        output_columns.append(config.record_key_col)
    
    if config.target_col not in df.columns:
        raise ValueError(
            f"Target column '{config.target_col}' is missing from the feature engineering dataframe."
        )
    
    output_columns.append(config.target_col)
    
    output_columns.extend(feature_columns)
    
    if "trip_distance_unit" in df.columns:
        output_columns.append("trip_distance_unit")
    
    output_columns = list(dict.fromkeys([
        col for col in output_columns
        if col in df.columns
    ]))
    
    feature_df = df[output_columns].copy()
    
    for col in feature_df.columns:
        if str(feature_df[col].dtype) == "Int64":
            feature_df[col] = feature_df[col].astype(float)
    
    print("Final feature dataset shape:", feature_df.shape)
    
    print("\nFeature columns for modeling later:")
    print(feature_columns)
    
    print("\nFirst rows of feature dataset:")
    display(feature_df.head())
    
    add_report(report, step, "final_feature_dataset_rows", len(feature_df))
    add_report(report, step, "final_feature_dataset_columns", feature_df.shape[1])
    add_report(report, step, "model_feature_columns_count", len(feature_columns))
    add_report(report, step, "model_feature_columns", str(feature_columns))
    
    add_report(
        report,
        step,
        "excluded_for_leakage",
        "start_time,end_time",
        "start_time is transformed into time features; end_time is excluded because it leaks target duration."
    )
    
    return feature_df, feature_columns

In [30]:
def build_feature_engineering_summary(
    original_df : pd.DataFrame,
    feature_df : pd.DataFrame,
    feature_columns : List[str],
    coordinate_system: str,
    report: List[Dict],
    config : FeatureEngineeringConfig
)-> List[str]:
    step = 'Step 7:Feature engineering summary report'
    
    missing_final = feature_df.isna().sum().sort_values(ascending=False)
    
    
    summary_lines : list[str] = []
    summary_lines.append('='*80)
    summary_lines.append('Phase 2 Feature Engineering Summary')
    summary_lines.append('='*80)
    summary_lines.append('')
    summary_lines.append(f'Input file:{config.input_file}')
    summary_lines.append(f'Output file :{config.engineered_output_file}')
    summary_lines.append('')
    summary_lines.append(f'Input rows : {len(original_df):,}')
    summary_lines.append(f'Input columns:{original_df.shape[1]}')
    summary_lines.append(f"Feature dataset rows: {len(feature_df):,}")
    summary_lines.append(f'Model feature columns:{len(feature_columns):,}')
    summary_lines.append('')
    summary_lines.append('Important leakage decision:')
    summary_lines.append('-end_time is not included as a modeling feature')
    summary_lines.append('- Speed is not created as feature because it depends on elapsed_seconds')
    summary_lines.append('')
    summary_lines.append('Feature columns:')
    for col in feature_columns:
        summary_lines.append(f'-{col}')
        
    summary_lines.append('')
    summary_lines.append('Remaning missing valuse in final feature dataset: ')
    
    if missing_final.sum() == 0:
        summary_lines.append('-No missing  valuse found')
    
    else:
        for col , count in missing_final.items():
            if int(count)>0:
                pct =100 *int(count)/max(len(feature_df),1)
                summary_lines.append(f'-{col}:{int(count):,} missing ({pct:.2f}%)')
                
    summary_lines.append('')
    summary_lines.append('final data types')
    for col, dtype in feature_df.dtypes.items():
        summary_lines.append(f'- {col}:{dtype}')
        
    add_report(report,step,'summary_created',True)
    return summary_lines


In [31]:
def save_feature_engineering_outputs(
    feature_df: pd.DataFrame,
    report : List[Dict],
    summary_lines : List[str],
    feature_columns : List[str],
    categorical_mapping:Dict,
    config : FeatureEngineeringConfig
) -> Tuple[pd.DataFrame,pd.DataFrame]:
    
    step = 'Step 8 : Save output files'
    
    feature_df.to_csv(config.engineered_output_file,index=False)
    
    report_df = pd.DataFrame(report)
    report_df.to_csv(config.report_csv_file,index=False)
    
    with open(config.report_txt_file,'w',encoding='utf-8') as f:
        f.write('\n'.join(summary_lines))
        f.write('\n')
        
    with open(config.feature_columns_file,'w',encoding='utf-8') as  f:
        for col in feature_columns:
            f.write(f'{col}\n')
            
    with open(config.categorical_mapping_file,'w',encoding='utf-8') as f:
        json.dump(categorical_mapping,f,ensure_ascii=False,indent=2)
        
    print('\nSaved Phase 2 output files:')
    print(f'-Engineered features dataset: {config.engineered_output_file}')
    print(f'-Feature engineering report CSV:{config.report_csv_file}')
    print(f'Feature engineering report TXT: {config.report_txt_file}')
    print(f'-Feature columns list :{config.feature_columns_file}')
    print(f'- Feature columns list:{config.feature_columns_file}')
    print(f'- Categorical mapping JSON : {config.categorical_mapping_file}')
    
    
    add_report(report,step,'outputs_saved',True)
    return feature_df , report_df

In [32]:
def run_feature_engineering_pipeline(config:FeatureEngineeringConfig) -> Tuple[pd.DataFrame,pd.DataFrame]:
    report : List[Dict] = []
    
    print('Starting Phase 2 feature engineering pipeline....')
    
    df = load_and_inspect_cleaned_data(config,report)
    original_df = df.copy()
    
    
    df = validate_cleaned_data(df,config,report)
    df,time_features = add_time_features(df,config,report)
    df, distance_features , coordinate_system = add_distance_features(df,config,report)
    df,categorical_features,categorical_mapping = add_categorical_features(df,config,report)
    
    feature_df , feature_columns = build_feature_dataset(
        df = df,
        config= config,
        time_features= time_features,
        distance_features= distance_features,
        categorical_features=categorical_features,
        report= report
    )
    
    summary_lines = build_feature_engineering_summary(
        original_df=original_df,
        feature_df= feature_df,
        feature_columns=  feature_columns,
        coordinate_system= coordinate_system,
        report= report,
        config=  config
    )
    
    
    feature_df ,report_df = save_feature_engineering_outputs(
        feature_df= feature_df,
        report=report,
        summary_lines=summary_lines,
        feature_columns=feature_columns,
        categorical_mapping=categorical_mapping,
        config= config
    )
    
    print('\n Phase 2 feature engineering completed successfully.')
    print('Final feature dataset shape:', feature_df.shape)
    
    return feature_df,report_df


In [33]:
INPUT_FILE = 'cleaned_student_version.csv'

config = FeatureEngineeringConfig(
    input_file = INPUT_FILE,
    engineered_output_file='feature_engineered_student_version.csv',
    report_csv_file= 'feature_engineering_report.csv',
    report_txt_file= 'feature_engineering_report.txt',
    feature_columns_file='feature_columns.txt',
    categorical_mapping_file='categorical_feature_mapping.json',
    rare_category_min_count=10,
    morning_rush_start=7,
    morning_rush_end=10,
    evening_rush_start=16,
    evening_rush_end=19,
    create_cyclical_time_features=True,
    one_hot_encode_categoricals =True,
    include_raw_coordinates_as_features= True,
    include_record_key_in_output=True
)

feature_df,feature_report_df = run_feature_engineering_pipeline(config)


Starting Phase 2 feature engineering pipeline....
Loaded Cleaned DataSet:
Shape: (1045079, 11)

Columns:


,Columns
0,record_key
1,carrier_code
2,start_time
3,end_time
4,rider_total
5,start_xcoord
6,start_ycoord
7,end_xcoord
8,end_ycoord
9,save_forward_marker



Data Types


record_key              object
carrier_code             int64
start_time              object
end_time                object
rider_total              int64
start_xcoord           float64
start_ycoord           float64
end_xcoord             float64
end_ycoord             float64
save_forward_marker     object
elapsed_seconds          int64
dtype: object


 First rows


...

save_forward_marker_clean value counts:


save_forward_marker_clean
2      559171
YES    485908
Name: count, dtype: Int64

Created Step 5:  Categorical feature preparation one_hot_encoded_featurs: 4 ['carrier_code_clean_1', 'carrier_code_clean_2', 'save_forward_marker_clean_2', 'save_forward_marker_clean_YES']
['carrier_code_clean_1', 'carrier_code_clean_2', 'save_forward_marker_clean_2', 'save_forward_marker_clean_YES']
Final feature dataset shape: (1045079, 25)

Feature columns for modeling later:
['rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'start_hour', 'start_day_of_week', 'start_month', 'is_weekend', 'is_rush_hour', 'start_hour_sin', 'start_hour_cos', 'start_day_of_week_sin', 'start_day_of_week_cos', 'start_month_sin', 'start_month_cos', 'same_start_end_location', 'trip_distance_projected', 'trip_distance', 'carrier_code_clean_1', 'carrier_code_clean_2', 'save_forward_marker_clean_2', 'save_forward_marker_clean_YES']

First rows of feature dataset:


,record_key,elapsed_seconds,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,start_hour,start_day_of_week,start_month,is_weekend,is_rush_hour,start_hour_sin,start_hour_cos,start_day_of_week_sin,start_day_of_week_cos,start_month_sin,start_month_cos,same_start_end_location,trip_distance_projected,trip_distance,carrier_code_clean_1,carrier_code_clean_2,save_forward_marker_clean_2,save_forward_marker_clean_YES
0,rid2875421,455,1,-73.982155,40.767937,-73.964630,40.765602,17,0,3,0.0,1.0,-0.965926,-0.258819,0.000000,1.000000,1.000000e+00,6.123234e-17,0.0,0.017680,0.017680,0,1,1,0
1,rid2377394,663,1,-73.980415,40.738564,-73.999481,40.731152,0,6,6,1.0,0.0,0.000000,1.000000,-0.781831,0.623490,1.224647e-16,-1.000000e+00,0.0,0.020456,0.020456,1,0,0,1
2,rid3858529,2124,1,-73.979027,40.763939,-74.005333,40.710087,11,1,1,0.0,0.0,0.258819,-0.965926,0.781831,0.623490,5.000000e-01,8.660254e-01,0.0,0.059934,0.059934,0,1,1,0
3,rid3504673,429,1,-74.010040,40.719971,-74.012268,40.706718,19,2,4,0.0,0.0,-0.965926,0.258819,0.974928,-0.222521,8.660254e-01,-5.000000e-01,0.0,0.013438,0.013438,0,1,1,0
4,rid2181028,435,1,-73.973053,40.793209,-73.972923,40.782520,13,5,3,1.0,0.0,-0.258819,-0.965926,-0.974928,-0.222521,1.000000e+00,6.123234e-17,0.0,0.010690,0.010690,0,1,1,0



Saved Phase 2 output files:
-Engineered features dataset: feature_engineered_student_version.csv
-Feature engineering report CSV:feature_engineering_report.csv
Feature engineering report TXT: feature_engineering_report.txt
-Feature columns list :feature_columns.txt
- Feature columns list:feature_columns.txt
- Categorical mapping JSON : categorical_feature_mapping.json

 Phase 2 feature engineering completed successfully.
Final feature dataset shape: (1045079, 25)


In [34]:
df_fe = pd.read_csv("feature_engineered_student_version.csv")
print(df_fe.shape)

with open("feature_columns.txt", "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f if line.strip()]

print(feature_cols)

(1045079, 25)
['rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'start_hour', 'start_day_of_week', 'start_month', 'is_weekend', 'is_rush_hour', 'start_hour_sin', 'start_hour_cos', 'start_day_of_week_sin', 'start_day_of_week_cos', 'start_month_sin', 'start_month_cos', 'same_start_end_location', 'trip_distance_projected', 'trip_distance', 'carrier_code_clean_1', 'carrier_code_clean_2', 'save_forward_marker_clean_2', 'save_forward_marker_clean_YES']


## Phase 3: Modeling and Evaluation


In [60]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.svm import LinearSVR, SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

In [ ]:
@dataclass
class ModelingConfig:
    input_file : str = 'feature_engineered_student_version.csv'
    feature_columns_file : str = 'feature_columns.txt'
    target_col : str = 'elapsed_seconds'
    record_key_col : str = 'record_key'
    test_size : float = 0.20
    random_state : int = 6
    
    
    model_comparison_file : str = 'model_comparison.csv'
    test_predictions_file : str = 'test_predictions.csv'
    modeling_report_file : str = 'modeling_report.txt'
    best_model_file : str = 'best_model.pkl'
    final_feature_columns_file: str = 'final_model_feature_columns.txt'
    feature_importance_file : str = 'best_model_feature_importance_or_coefficients.csv'
    
    run_cross_validation : bool = True
    cv_folds : int = 5
    
    
    knn_neighbors : Tuple[int,...]= (3,5,11)
    decision_tree_depths : tuple[Optional[int],...] = (5,10,None)
    random_forest_n_estimators : int = 150
    random_forest_max_depth : Optional[int] = None
    
    
    include_rbf_svr : bool = True
    max_rows_for_rbf_svr : int = 15000
    
    polynomial_degree  : int = 2
    max_polynomial_features : int =12
    
    
    

    run_optional_classfication_demo : bool = False
    duration_class_labels : Tuple[str,str,str]= ('short','medium','long')
    

In [ ]:
def safe_read_csv(path: str) -> pd.DataFrame:
    file_path  = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(
            f'File not found : {file_path.resolve()}'
            "please run Phase 2 and make sure you have feature_engineered_student_version.csv"
        )
        
    return  pd.read_csv(file_path)



def load_feature_columns(path:str) -> List[str]:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(
            f'Feature columns file not found {file_path.resolve()}'
            "please run Phase 2  and make sure you have feature_columns.txt"
        )
        
    with open(file_path,'r',encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]
    
    
def rmse_score(y_true,y_pred)-> float:
    return float(np.sqrt(mean_squared_error(y_true,y_pred)))


def regression_metrics(y_true,y_pred)-> Dict[str,float]:
    mae = float(mean_absolute_error(y_true,y_pred))
    rmse = rmse_score(y_true,y_pred)
    r2 = float(r2_score(y_true,y_pred))
    
    return{
        'MAE_seconds':mae,
        'RMSE_seconds':rmse,
        'R2' : r2,
        'MAE_minutes' : mae/60.0,
        'RMSE_minutes': rmse/60.0
    }
    
def make_regression_pipeline(model,scale_numeric:bool = False) -> Pipeline:
    steps = [('imputer',SimpleImputer(strategy='median'))]
    if scale_numeric:
        steps.append(('scaler',StandardScaler()))
    steps.append(('model',model))
    return Pipeline(steps)


def clean_model_name(key:str) -> str:
    return key.replace('_',' ').title()



def select_polynomial_base_features(X:pd.DataFrame,config:ModelingConfig)-> List[str]:
    

    
    priority_keywords = [
        'trip_distance',
        'rider_total',
        'start_hour',
        'start_day_of_week',
        'start_month',
        'xcoord',
        'ycoord'
    ]
    
    selected = []
    for col in X.columns:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in priority_keywords):
            
            selected.append(col)
        
    selected = [col for col in selected if pd.api.types.is_numeric_dtype(X[col])]
    selected = list(dict.fromkeys(selected))[:config.max_polynomial_features]
    
    if not selected:
        selected = list(X.columns[:min(config.max_polynomial_features,X.shape[1])])
    
    return selected

In [38]:
def load_modeling_data(config:ModelingConfig) -> Tuple[pd.DataFrame,List[str]]:
    df = safe_read_csv(config.input_file)
    feature_columns = load_feature_columns(config.feature_columns_file)
    
    print('Loaded engineered dataset')
    print('Shape: ', df.shape)
    
    print('first rows:')
    display(df.head())
    
    print('loaded feature columns:')
    print(feature_columns)
    
    if config.target_col not in df.columns:
        raise ValueError(f'Target column {config.target_col} is missing')
    return df,feature_columns


In [39]:
def load_modeling_data(config:ModelingConfig)->Tuple[pd.DataFrame,List[str]]:
    df = safe_read_csv(config.input_file)
    feature_columns = load_feature_columns(config.feature_columns_file)
    
    print('Loaded engineering dataset')
    print('Shape; ', df.shape)
    
    print('\n First rows')
    display(df.head())
    
    print('\n Feature columns loaded from file: ')
    print(feature_columns)
    
    if config.target_col not in df.columns:
        raise ValueError(f'Target column {config.target_col} is missing')
    
    return df,feature_columns

In [ ]:
def prepare_model_inputs(
    df: pd.DataFrame,
    feature_columns: List[str],
    config: ModelingConfig
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, List[str]]:
    
    df = df.copy()
    
    print("\nPreparing model inputs...")
    print("Input dataframe shape:", df.shape)
    
    if df.empty:
        raise ValueError(
            "feature_engineered_student_version.csv is empty. "
            "Run Phase 1 and Phase 2 again and make sure the engineered file has rows."
        )
    
    if config.target_col not in df.columns:
        raise ValueError(f"Target column '{config.target_col}' is missing.")
    
    existing_features = [col for col in feature_columns if col in df.columns]
    missing_features = [col for col in feature_columns if col not in df.columns]
    
    print("Feature columns from file:", len(feature_columns))
    print("Existing feature columns:", len(existing_features))
    print("Missing feature columns:", len(missing_features))
    
    if missing_features:
        print("Missing features:")
        print(missing_features)
    
    forbidden_cols = {
        config.target_col,
        config.record_key_col,
        "trip_distance_unit",
        "start_time",
        "end_time",
    }
    
    existing_features = [col for col in existing_features if col not in forbidden_cols]
    
    if len(existing_features) == 0:
        raise ValueError(
            "No valid feature columns remained after removing leakage/non-model columns. "
            "Check feature_columns.txt."
        )
    
    X_candidate = df[existing_features].copy()
    
    for col in X_candidate.columns:
        if not pd.api.types.is_numeric_dtype(X_candidate[col]):
            X_candidate[col] = (
                X_candidate[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .replace({"nan": np.nan, "None": np.nan, "<NA>": np.nan})
            )
            X_candidate[col] = pd.to_numeric(X_candidate[col], errors="coerce")
    
    final_features = [
        col for col in X_candidate.columns
        if pd.api.types.is_numeric_dtype(X_candidate[col])
    ]
    
    print("Numeric usable features:", len(final_features))
    
    if len(final_features) == 0:
        raise ValueError(
            "No numeric usable features remained. "
            "Check Feature Engineering output and feature_columns.txt."
        )
    
    X = X_candidate[final_features].copy()
    
    y = (
        df[config.target_col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "<NA>": np.nan})
    )
    
    y = pd.to_numeric(y, errors="coerce")
    
    valid_target_mask = y.notna() & np.isfinite(y)
    
    print("Rows before target validation:", len(df))
    print("Valid target rows:", int(valid_target_mask.sum()))
    print("Invalid target rows:", int((~valid_target_mask).sum()))
    
    if int(valid_target_mask.sum()) == 0:
        raise ValueError(
            "No valid target rows remained. "
            "The elapsed_seconds column is empty, non-numeric, or the engineered dataset has zero rows."
        )
    
    X = X.loc[valid_target_mask].copy()
    y = y.loc[valid_target_mask].copy()
    modeling_df = df.loc[valid_target_mask].copy()
    
    if len(X) == 0:
        raise ValueError(
            "X has zero rows after preparation. "
            "Check feature_engineered_student_version.csv and feature_columns.txt."
        )
    
    print("\nPrepared model inputs successfully.")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Final features:", final_features)
    
    print("\nTop missing values in X:")
    display(X.isna().sum().sort_values(ascending=False).head(30))
    
    return X, y, modeling_df, final_features

In [41]:
def split_train_test(
    X : pd.DataFrame,
    y : pd.Series,
    modeling_df : pd.DataFrame,
    config : ModelingConfig
):
    if len(X) == 0:
        raise ValueError('X has zero rows. Cannot perform train/test split')
    
    if len(y) == 0:
        raise ValueError('y has zero rows , Cannot perform train/test split.')
    
    if len(X) != len(y):
        raise ValueError(
            f'X and y hase different lenghts : X={len(X)}, y={len(y)}'
        )
        
        
    if len(modeling_df) != len(X):
        raise ValueError(
            f'modeling_df and X have different lenghts : Modeling_df={len(modeling_df)}, X={len(X)}'
        )
        
    X_train,X_test,y_train,y_test , meta_train,meta_test = train_test_split(
        X,
        y,
        modeling_df,
        test_size= config.test_size,
        random_state=config.random_state
    )
    
    
    print('Train/test split completed')
    print('X_train:', X_train.shape)
    print('X_test: ',X_test.shape)
    print('y_train: ',y_train.shape)
    print('y_test: ',y_test.shape)
    print('meta_train:', meta_train.shape)
    print('meta_test:', meta_test.shape)
    
    
    return X_train,X_test,y_train,y_test,meta_train,meta_test


In [ ]:
def build_regression_models(
    X_train: pd.DataFrame,
    config: ModelingConfig
) -> Dict[str, Pipeline]:
    
    models: Dict[str, Pipeline] = {}
    
    polynomial_degree = getattr(
        config,
        "polynomial_degree",
        getattr(config, "polynomail_degree", 2)
    )
    
    models["baseline_dummy_median"] = make_regression_pipeline(
        DummyRegressor(strategy="median"),
        scale_numeric=False,
    )
    
    models["linear_regression"] = make_regression_pipeline(
        LinearRegression(),
        scale_numeric=True,
    )
    
    polynomial_features = select_polynomial_base_features(X_train, config)
    
    passthrough_features = [
        col for col in X_train.columns
        if col not in polynomial_features
    ]
    
    polynomial_preprocessor = ColumnTransformer(
        transformers=[
            (
                "poly",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("poly", PolynomialFeatures(
                        degree=polynomial_degree,
                        include_bias=False
                    )),
                ]),
                polynomial_features,
            ),
            (
                "rest",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                passthrough_features,
            ),
        ],
        remainder="drop"
    )
    
    models[f"polynomial_regression_degree_{polynomial_degree}"] = Pipeline([
        ("preprocessor", polynomial_preprocessor),
        ("model", Ridge(alpha=1.0, random_state=config.random_state)),
    ])
    
    for depth in config.decision_tree_depths:
        depth_name = "none" if depth is None else str(depth)
        
        models[f"decision_tree_depth_{depth_name}"] = make_regression_pipeline(
            DecisionTreeRegressor(
                max_depth=depth,
                random_state=config.random_state,
                min_samples_leaf=10
            ),
            scale_numeric=False,
        )
    
    models["random_forest_tuned"] = make_regression_pipeline(
        RandomForestRegressor(
            n_estimators=config.random_forest_n_estimators,
            max_depth=22,
            random_state=config.random_state,
            n_jobs=-1,
            min_samples_leaf=10,
            max_features="sqrt"
        ),
        scale_numeric=False,
    )
    
    models["extra_trees"] = make_regression_pipeline(
        ExtraTreesRegressor(
            n_estimators=80,
            max_depth=24,
            random_state=config.random_state,
            n_jobs=-1,
            min_samples_leaf=10,
            max_features="sqrt"
        ),
        scale_numeric=False,
    )
    
    models["hist_gradient_boosting"] = make_regression_pipeline(
        HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.06,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=0.10,
            random_state=config.random_state
        ),
        scale_numeric=False,
    )
    

    hist_gb_base = make_regression_pipeline(
        HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.06,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=0.10,
            random_state=config.random_state
        ),
        scale_numeric=False,
    )
    
    models["hist_gradient_boosting_log_target"] = TransformedTargetRegressor(
        regressor=hist_gb_base,
        func=np.log1p,
        inverse_func=np.expm1
    )
    
  
    models["linear_svr"] = make_regression_pipeline(
        LinearSVR(
            random_state=config.random_state,
            max_iter=10000
        ),
        scale_numeric=True,
    )
    

    
    print("Regression models prepared:")
    print(list(models.keys()))
    
    print("\nPolynomial regression base features:")
    print(polynomial_features)
    
    return models

In [ ]:
def evaluate_regression_models(
    models: Dict[str, Pipeline],
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    config: ModelingConfig,
) -> Tuple[pd.DataFrame, Dict[str, Pipeline]]:
    
    results = []
    fitted_models: Dict[str, Pipeline] = {}
    
    max_rows_for_cv = getattr(config, "max_rows_for_cross_validation", 100000)
    
    for model_key, model in models.items():
        
        print(f"\nTraining model: {clean_model_name(model_key)}")
        
        model.fit(X_train, y_train)
        fitted_models[model_key] = model
        
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        
        train_metrics = regression_metrics(y_train, train_pred)
        test_metrics = regression_metrics(y_test, test_pred)
        
        row = {
            "model_key": model_key,
            "model_name": clean_model_name(model_key),
            
            "train_MAE_seconds": train_metrics["MAE_seconds"],
            "train_RMSE_seconds": train_metrics["RMSE_seconds"],
            "train_R2": train_metrics["R2"],
            
            "test_MAE_seconds": test_metrics["MAE_seconds"],
            "test_RMSE_seconds": test_metrics["RMSE_seconds"],
            "test_R2": test_metrics["R2"],
            
            "test_MAE_minutes": test_metrics["MAE_minutes"],
            "test_RMSE_minutes": test_metrics["RMSE_minutes"],
        }
        
        if config.run_cross_validation and len(X_train) <= max_rows_for_cv:
            try:
                print(f"Running {config.cv_folds}-fold cross-validation for {model_key}...")
                
                cv_result = cross_validate(
                    model,
                    X_train,
                    y_train,
                    cv=config.cv_folds,
                    scoring={
                        "mae": "neg_mean_absolute_error",
                        "rmse": "neg_root_mean_squared_error",
                        "r2": "r2",
                    },
                    n_jobs=-1,
                    error_score=np.nan,
                )
                
                row["cv_MAE_seconds_mean"] = float(-np.nanmean(cv_result["test_mae"]))
                row["cv_RMSE_seconds_mean"] = float(-np.nanmean(cv_result["test_rmse"]))
                row["cv_R2_mean"] = float(np.nanmean(cv_result["test_r2"]))
                
            except Exception as e:
                row["cv_MAE_seconds_mean"] = np.nan
                row["cv_RMSE_seconds_mean"] = np.nan
                row["cv_R2_mean"] = np.nan
                row["cv_error"] = str(e)
        
        elif config.run_cross_validation and len(X_train) > max_rows_for_cv:
            row["cv_MAE_seconds_mean"] = np.nan
            row["cv_RMSE_seconds_mean"] = np.nan
            row["cv_R2_mean"] = np.nan
            row["cv_error"] = (
                f"Cross-validation skipped because X_train has {len(X_train):,} rows. "
                f"Limit is {max_rows_for_cv:,} rows."
            )
        
        results.append(row)
        
        print(
            f"Test MAE: {row['test_MAE_seconds']:.2f}s "
            f"({row['test_MAE_minutes']:.2f} min) | "
            f"Test RMSE: {row['test_RMSE_seconds']:.2f}s "
            f"({row['test_RMSE_minutes']:.2f} min) | "
            f"Test R2: {row['test_R2']:.4f}"
        )
    
    results_df = pd.DataFrame(results)
    
    results_df["rank_by_RMSE"] = results_df["test_RMSE_seconds"].rank(method="min", ascending=True)
    results_df["rank_by_MAE"] = results_df["test_MAE_seconds"].rank(method="min", ascending=True)
    results_df["combined_rank_RMSE_MAE"] = (
        results_df["rank_by_RMSE"] + results_df["rank_by_MAE"]
    )
    
    results_df = results_df.sort_values(
        ["combined_rank_RMSE_MAE", "test_RMSE_seconds"],
        ascending=True
    ).reset_index(drop=True)
    
    print("\nModel comparison sorted by combined RMSE + MAE rank:")
    display(results_df)
    
    best_by_rmse = results_df.sort_values("test_RMSE_seconds", ascending=True).iloc[0]
    best_by_mae = results_df.sort_values("test_MAE_seconds", ascending=True).iloc[0]
    best_by_combined = results_df.sort_values("combined_rank_RMSE_MAE", ascending=True).iloc[0]
    
    print("\nBest model by RMSE:")
    print(best_by_rmse["model_name"])
    print(f"RMSE: {best_by_rmse['test_RMSE_seconds']:.2f}s | MAE: {best_by_rmse['test_MAE_seconds']:.2f}s")
    
    print("\nBest model by MAE:")
    print(best_by_mae["model_name"])
    print(f"MAE: {best_by_mae['test_MAE_seconds']:.2f}s | RMSE: {best_by_mae['test_RMSE_seconds']:.2f}s")
    
    print("\nBest model by combined RMSE + MAE rank:")
    print(best_by_combined["model_name"])
    print(f"MAE: {best_by_combined['test_MAE_seconds']:.2f}s | RMSE: {best_by_combined['test_RMSE_seconds']:.2f}s")
    
    return results_df, fitted_models

In [44]:
def select_best_regression_model(
    result_df : pd.DataFrame,
    fitted_models: Dict[str,pd.Series]
)-> Tuple[str,Pipeline,pd.Series]:
    if result_df.empty:
        raise ValueError('No model results available.')
    
    best_row = result_df.iloc[0]
    best_key = best_row['model_key']
    best_model = fitted_models[best_key]
    
    print('Best regression model: ')
    print(best_model[['model_name','test_MAE_seconds','test_RMSE_seconds','test_R2']])
    
    return best_key,best_model,best_row

In [45]:

def extract_best_model_interpretation(
    best_model:Pipeline,
    feature_columns: List[str],
)-> pd.DataFrame:
    
    if 'model' not in best_model.named_steps:
        print('Best model pipeline does not have a direct model step for interpretation')
        return pd.DataFrame()
    
    model = best_model.named_steps['model']
    
    if hasattr(model,'feature_importances_') and len(model.feature_importances_) == len(feature_columns):
        interpretation_df = pd.DataFrame({
            'feature': feature_columns,
            'importance': model.feature_importance_,
        }).sort_values('importance',ascending=False)
        
        print('Top feature importances:')
        display(interpretation_df.head(30))
        return interpretation_df
    
    if hasattr(model,'coef_') and len(np.reval(model.coef_)) == len(feature_columns):
        coef = np.ravel(model.coef_)
        interpretation_df = pd.DataFrame({
            'feature':feature_columns,
            'coefficient': coef,
            'abs_coefficient': np.abs(coef),
        }).sort_values('abs_coefficient', ascending=False)
        
        
        print('Top coefficients:  ')
        display(interpretation_df.head(30))
        return interpretation_df
    
    print('Feature importance or coefficeints are not directly available for this best model.')
    return pd.DataFrame()

In [46]:
def save_modeling_outputs(
    results_df : pd.DataFrame,
    best_key : str,
    best_model :Pipeline,
    best_row : pd.Series,
    interpretation_df : pd.DataFrame,
    X_test : pd.DataFrame,
    y_test:pd.Series,
    meta_test :pd.DataFrame,
    final_feature_columns: List[str],
    config:ModelingConfig,
) -> pd.DataFrame:
    
    results_df.to_csv(config.model_comparison_file,index=False)
    
    with open(config.final_feature_columns_file,'w',encoding='utf-8') as f:
        for col in final_feature_columns:
            f.write(f'{col}\n')
            
    with open(config.best_model_file,'wb') as f:
        pickle.dump(best_model,f)
        
    y_pred = best_model.predict(X_test)
    prediction_df = pd.DataFrame({
        'actual_elapsed_seconds': y_test.valuse,
        'predicted_elapsed_seconds': y_pred,
        'absolute_error_seconds': np.abs(y_test.value - y_pred),
    }, index=y_test.index)
    
    prediction_df['absolute_error_minutes'] = prediction_df['absolute_error_seconds'] / 60.0
    
    if config.record_key_col in meta_test.columns:
        prediction_df.insert(0,config.record_key_col,meta_test[config.record_key_col].values)
        
    prediction_df.to_csv(config.test_predictions_file,index=False)
    
    interpretation_saved = False
    if not interpretation_df.empty:
        interpretation_df.to_csv(config.feature_importance_file,index=False)
        interpretation_saved = True
        
    lines = []
    lines.append('-'*60)
    lines.append('Phase 3 Modeling and Evaluation Report')
    lines.append('-'*60)
    lines.append('')
    lines.append(f'Input file:{config.input_file}')
    lines.append(f'Target column:{config.input_file}')
    lines.append(f'Test size: {config.test_size}')
    lines.append(f'Random state: {config.random_state}')
    lines.append('')
    lines.append('Main task : Regression')
    lines.append('Fianl model selection metric : lowest test RMSE')
    lines.append('')
    lines.append('Best model:')
    lines.append(f'- Model key: {best_key}')
    lines.append(f'-Model name: {best_row['model_name']}')
    lines.append(f'- Test MAE: {best_row['test_MAE_seconds']:.2f} seconds ({best_row['test_MAE_minutes']:.2f}) minutes')
    lines.append(f'- Test RMSE : {best_row['test_RMSE_seconds']:.2f} seconds ({best_row['test_RMSE_minutes']:.2f} minutes)')
    lines.append('')
    lines.append('Model comparison: ')
    
    for _ , row in results_df.iterrows():
        lines.append(
            f'- {row['model_name']}: '
            f'MAE={row['test_MAE_seconds']:.2f}s'
            f'RMSE={row['test_RMSE_seconds']:.2f}s'
            f'R^2{row['test_R2']:.4f}'
        )
        
    lines.append('')
    lines.append('Imortant course alignment note: ')
    lines.append('- KNN, Decision Tree, Linear Regression, Polynomial Regression, Random Forest, and SVM were used as regression models.')
    lines.append('')
    lines.append('Output files:')
    lines.append(f'- {config.model_comparison_file}')
    lines.append(f'- {config.test_predictions_file}')
    lines.append(f'- {config.best_model_file}')
    lines.append(f'- {config.final_feature_columns_file}')
    if interpretation_saved:
        lines.append(f'-{config.feature_importance_file}')
    
    with open(config.modeling_report_file,'w',encoding='utf-8') as f:
        f.write('\n'.join(lines))
        f.write('\n')
        
        
    print('saved modeling outputs:')
    print(f'- {config.model_comparison_file}')
    print(f'- {config.test_predictions_file}')
    print(f'- {config.best_model_file}')
    print(f'- {config.best_model_file}')
    print(f'- {config.final_feature_columns_file}')
    
    if interpretation_saved:
        print(f'- {config.feature_importance_file}')
        
    return prediction_df


In [47]:
# def create_duration_classes(y: pd.Series, labels: Tuple[str, str, str]) -> pd.Series:
#     # qcut creates balanced bins when possible.
#     try:
#         return pd.qcut(y, q=3, labels=labels, duplicates='drop').astype(str)
#     except Exception:
#         q1, q2 = y.quantile([0.33, 0.66])
#         return pd.cut(
#             y,
#             bins=[-np.inf, q1, q2, np.inf],
#             labels=labels,
#         ).astype(str)


# def run_optional_classification_demo(
#     X: pd.DataFrame,
#     y: pd.Series,
#     config: ModelingConfig,
# ) -> pd.DataFrame:
#     if not config.run_optional_classification_demo:
#         print('Optional classification demo is disabled.')
#         return pd.DataFrame()

#     y_class = create_duration_classes(y, config.duration_class_labels)

#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y_class,
#         test_size=config.test_size,
#         random_state=config.random_state,
#         stratify=y_class if y_class.nunique() > 1 else None,
#     )

#     classifiers = {
#         'logistic_regression_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler()),
#             ('model', LogisticRegression(max_iter=3000, random_state=config.random_state)),
#         ]),
#         'naive_bayes_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler()),
#             ('model', GaussianNB()),
#         ]),
#         'knn_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler()),
#             ('model', KNeighborsClassifier(n_neighbors=5)),
#         ]),
#         'decision_tree_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('model', DecisionTreeClassifier(max_depth=10, random_state=config.random_state)),
#         ]),
#         'random_forest_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('model', RandomForestClassifier(n_estimators=100, random_state=config.random_state, n_jobs=-1)),
#         ]),
#         'svm_classifier': Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler()),
#             ('model', SVC(kernel='rbf', random_state=config.random_state)),
#         ]),
#     }

#     results = []

#     for key, clf in classifiers.items():
#         print(f'\nTraining optional classifier: {clean_model_name(key)}')
#         clf.fit(X_train, y_train)
#         pred = clf.predict(X_test)

#         precision, recall, f1, _ = precision_recall_fscore_support(
#             y_test,
#             pred,
#             average='weighted',
#             zero_division=0,
#         )

#         results.append({
#             'classifier_key': key,
#             'classifier_name': clean_model_name(key),
#             'accuracy': accuracy_score(y_test, pred),
#             'weighted_precision': precision,
#             'weighted_recall': recall,
#             'weighted_f1': f1,
#         })

#     classification_results_df = pd.DataFrame(results).sort_values('weighted_f1', ascending=False)

#     print('\nOptional classification demo results:')
#     display(classification_results_df)

#     classification_results_df.to_csv('optional_duration_classification_results.csv', index=False)
#     return classification_results_df

In [63]:
def run_modeling_pipeline(config: ModelingConfig):
    
    print("Starting Phase 3 Modeling and Evaluation pipeline...")
    
    df, feature_columns = load_modeling_data(config)
    
    X, y, modeling_df, final_feature_columns = prepare_model_inputs(
        df=df,
        feature_columns=feature_columns,
        config=config
    )
    
    X_train, X_test, y_train, y_test, meta_train, meta_test = split_train_test(
        X=X,
        y=y,
        modeling_df=modeling_df,
        config=config
    )
    
    models = build_regression_models(X_train, config)
    
    results_df, fitted_models = evaluate_regression_models(
        models=models,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        config=config
    )
    
    if results_df.empty:
        raise ValueError("No model results were produced.")
    
    selection_metric = getattr(config, "selection_metric", "combined").lower()
    
    if selection_metric == "rmse":
        best_row = results_df.sort_values("test_RMSE_seconds", ascending=True).iloc[0]
        selection_description = "lowest test RMSE"
    
    elif selection_metric == "mae":
        best_row = results_df.sort_values("test_MAE_seconds", ascending=True).iloc[0]
        selection_description = "lowest test MAE"
    
    else:
        best_row = results_df.sort_values(
            ["combined_rank_RMSE_MAE", "test_RMSE_seconds"],
            ascending=True
        ).iloc[0]
        selection_description = "best combined RMSE + MAE rank"
    
    best_model_key = best_row["model_key"]
    best_model = fitted_models[best_model_key]
    
    print("\nBest regression model:")
    print("Selection rule:", selection_description)
    print("Model key:", best_model_key)
    print("Model name:", best_row["model_name"])
    print(f"Test MAE: {best_row['test_MAE_seconds']:.2f} seconds ({best_row['test_MAE_minutes']:.2f} minutes)")
    print(f"Test RMSE: {best_row['test_RMSE_seconds']:.2f} seconds ({best_row['test_RMSE_minutes']:.2f} minutes)")
    print(f"Test R2: {best_row['test_R2']:.4f}")
    
    test_pred = best_model.predict(X_test)
    
    test_predictions_df = pd.DataFrame()
    
    if config.record_key_col in meta_test.columns:
        test_predictions_df[config.record_key_col] = meta_test[config.record_key_col].values
    
    test_predictions_df["actual_elapsed_seconds"] = y_test.values
    test_predictions_df["predicted_elapsed_seconds"] = test_pred
    
    test_predictions_df["absolute_error_seconds"] = np.abs(
        test_predictions_df["actual_elapsed_seconds"] -
        test_predictions_df["predicted_elapsed_seconds"]
    )
    
    test_predictions_df["actual_elapsed_minutes"] = (
        test_predictions_df["actual_elapsed_seconds"] / 60.0
    )
    
    test_predictions_df["predicted_elapsed_minutes"] = (
        test_predictions_df["predicted_elapsed_seconds"] / 60.0
    )
    
    test_predictions_df["absolute_error_minutes"] = (
        test_predictions_df["absolute_error_seconds"] / 60.0
    )
    
    print("\nTest predictions sample:")
    display(test_predictions_df.head())
    
    results_df.to_csv(config.model_comparison_file, index=False)
    test_predictions_df.to_csv(config.test_predictions_file, index=False)
    
    with open(config.best_model_file, "wb") as f:
        pickle.dump(best_model, f)
    
    with open(config.final_feature_columns_file, "w", encoding="utf-8") as f:
        for col in final_feature_columns:
            f.write(col + "\n")
    
    try:
        save_feature_importance_or_coefficients(
            best_model=best_model,
            feature_columns=final_feature_columns,
            output_file=config.feature_importance_file
        )
    except Exception as e:
        print("Feature importance/coefficient export skipped:")
        print(str(e))
    
    optional_classification_results_df = None
    
    run_optional_classification = getattr(
        config,
        "run_optional_classification_demo",
        getattr(config, "run_optional_classfication_demo", False)
    )
    
    if run_optional_classification:
        try:
            optional_classification_results_df = run_optional_classification_demo(
                X_train=X_train,
                X_test=X_test,
                y_train=y_train,
                y_test=y_test,
                config=config
            )
        except Exception as e:
            print("Optional classification demo skipped because of error:")
            print(str(e))
    
    try:
        with open(config.modeling_report_file, "w", encoding="utf-8") as f:
            f.write("Phase 3 Modeling and Evaluation Report\n")
            f.write("=" * 45 + "\n\n")
            f.write(f"Input file: {config.input_file}\n")
            f.write(f"Rows: {len(df):,}\n")
            f.write(f"Features used: {len(final_feature_columns)}\n")
            f.write(f"Train rows: {len(X_train):,}\n")
            f.write(f"Test rows: {len(X_test):,}\n\n")
            
            f.write("Model selection:\n")
            f.write(f"Selection rule: {selection_description}\n")
            f.write(f"Best model key: {best_model_key}\n")
            f.write(f"Best model name: {best_row['model_name']}\n")
            f.write(f"Test MAE seconds: {best_row['test_MAE_seconds']:.4f}\n")
            f.write(f"Test RMSE seconds: {best_row['test_RMSE_seconds']:.4f}\n")
            f.write(f"Test R2: {best_row['test_R2']:.6f}\n\n")
            
            f.write("Interpretation:\n")
            f.write(
                "MAE shows the average absolute prediction error in seconds. "
                "RMSE penalizes large errors more strongly and is useful for detecting models "
                "that perform badly on extreme trips.\n\n"
            )
            
            f.write("All model comparison results:\n")
            f.write(results_df.to_string(index=False))
    
    except Exception as e:
        print("Modeling report text file was not saved:")
        print(str(e))
    
    print("\nSaved files:")
    print(config.model_comparison_file)
    print(config.test_predictions_file)
    print(config.best_model_file)
    print(config.final_feature_columns_file)
    print(config.modeling_report_file)
    
    print("\nPhase 3 completed successfully.")
    
    return results_df, test_predictions_df, best_model, optional_classification_results_df

In [64]:
config = ModelingConfig(
    input_file='feature_engineered_student_version.csv',
    feature_columns_file='feature_columns.txt',
    target_col='elapsed_seconds',
    record_key_col='record_key',
    test_size=0.20,
    random_state=6,
    
    model_comparison_file='model_comparison.csv',
    test_predictions_file='test_prediction.csv',
    modeling_report_file='modeling_report.txt',
    best_model_file='best_model.pkl',
    final_feature_columns_file='final_model_feature_columns.txt',
    feature_importance_file='best_model_feature_importance_or_coefficients.csv',
    
    run_cross_validation=False,
    cv_folds=5,
    
    knn_neighbors=(),
    decision_tree_depths=(5, 10, None),
    
    random_forest_n_estimators=50,
    random_forest_max_depth=None,
    
    include_rbf_svr=False,
    max_rows_for_rbf_svr=15000,
    
    polynomial_degree=2,
    max_polynomial_features=12,
    
    run_optional_classfication_demo=False
)

model_result_df, test_predictions_df, best_model, optional_classification_results_df = run_modeling_pipeline(config)

Starting Phase 3 Modeling and Evaluation pipeline...
Loaded engineering dataset
Shape;  (1045079, 25)

 First rows


,record_key,elapsed_seconds,rider_total,start_xcoord,start_ycoord,end_xcoord,end_ycoord,start_hour,start_day_of_week,start_month,is_weekend,is_rush_hour,start_hour_sin,start_hour_cos,start_day_of_week_sin,start_day_of_week_cos,start_month_sin,start_month_cos,same_start_end_location,trip_distance_projected,trip_distance,carrier_code_clean_1,carrier_code_clean_2,save_forward_marker_clean_2,save_forward_marker_clean_YES
0,rid2875421,455,1,-73.982155,40.767937,-73.964630,40.765602,17,0,3,0.0,1.0,-0.965926,-0.258819,0.000000,1.000000,1.000000e+00,6.123234e-17,0.0,0.017680,0.017680,0,1,1,0
1,rid2377394,663,1,-73.980415,40.738564,-73.999481,40.731152,0,6,6,1.0,0.0,0.000000,1.000000,-0.781831,0.623490,1.224647e-16,-1.000000e+00,0.0,0.020456,0.020456,1,0,0,1
2,rid3858529,2124,1,-73.979027,40.763939,-74.005333,40.710087,11,1,1,0.0,0.0,0.258819,-0.965926,0.781831,0.623490,5.000000e-01,8.660254e-01,0.0,0.059934,0.059934,0,1,1,0
3,rid3504673,429,1,-74.010040,40.719971,-74.012268,40.706718,19,2,4,0.0,0.0,-0.965926,0.258819,0.974928,-0.222521,8.660254e-01,-5.000000e-01,0.0,0.013438,0.013438,0,1,1,0
4,rid2181028,435,1,-73.973053,40.793209,-73.972923,40.782520,13,5,3,1.0,0.0,-0.258819,-0.965926,-0.974928,-0.222521,1.000000e+00,6.123234e-17,0.0,0.010690,0.010690,0,1,1,0



 Feature columns loaded from file: 
['rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'start_hour', 'start_day_of_week', 'start_month', 'is_weekend', 'is_rush_hour', 'start_hour_sin', 'start_hour_cos', 'start_day_of_week_sin', 'start_day_of_week_cos', 'start_month_sin', 'start_month_cos', 'same_start_end_location', 'trip_distance_projected', 'trip_distance', 'carrier_code_clean_1', 'carrier_code_clean_2', 'save_forward_marker_clean_2', 'save_forward_marker_clean_YES']

Preparing model inputs...
Input dataframe shape: (1045079, 25)
Feature columns from file: 23
Existing feature columns: 23
Missing feature columns: 0
Numeric usable features: 23
Rows before target validation: 1045079
Valid target rows: 1045079
Invalid target rows: 0

Prepared model inputs successfully.
X shape: (1045079, 23)
y shape: (1045079,)
Final features: ['rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'start_hour', 'start_day_of_week', 'start_month', 'is_weekend

rider_total                      0
start_day_of_week_sin            0
save_forward_marker_clean_2      0
carrier_code_clean_2             0
carrier_code_clean_1             0
trip_distance                    0
trip_distance_projected          0
same_start_end_location          0
start_month_cos                  0
start_month_sin                  0
start_day_of_week_cos            0
start_hour_cos                   0
start_xcoord                     0
start_hour_sin                   0
is_rush_hour                     0
is_weekend                       0
start_month                      0
start_day_of_week                0
start_hour                       0
end_ycoord                       0
end_xcoord                       0
start_ycoord                     0
save_forward_marker_clean_YES    0
dtype: int64

Train/test split completed
X_train: (836063, 23)
X_test:  (209016, 23)
y_train:  (836063,)
y_test:  (209016,)
meta_train: (836063, 25)
meta_test: (209016, 25)
Regression models prepared:
['baseline_dummy_median', 'linear_regression', 'polynomial_regression_degree_2', 'decision_tree_depth_5', 'decision_tree_depth_10', 'decision_tree_depth_none', 'random_forest_tuned', 'extra_trees', 'hist_gradient_boosting', 'hist_gradient_boosting_log_target', 'linear_svr']

Polynomial regression base features:
['rider_total', 'start_xcoord', 'start_ycoord', 'end_xcoord', 'end_ycoord', 'start_hour', 'start_day_of_week', 'start_month', 'start_hour_sin', 'start_hour_cos', 'start_day_of_week_sin', 'start_day_of_week_cos']

Training model: Baseline Dummy Median
Test MAE: 555.21s (9.25 min) | Test RMSE: 3144.17s (52.40 min) | Test R2: -0.0083

Training model: Linear Regression
Test MAE: 443.31s (7.39 min) | Test RMSE: 3105.67s (51.76 min) | Test R2: 0.0163

Training model: Polynomial Regression Degree 2
Tes

,model_key,model_name,train_MAE_seconds,train_RMSE_seconds,train_R2,test_MAE_seconds,test_RMSE_seconds,test_R2,test_MAE_minutes,test_RMSE_minutes,rank_by_RMSE,rank_by_MAE,combined_rank_RMSE_MAE
0,hist_gradient_boosting_log_target,Hist Gradient Boosting Log Target,317.455800,6336.792101,0.008561,300.964349,3076.373605,0.034723,5.016072,51.272893,1.0,1.0,2.0
1,random_forest_tuned,Random Forest Tuned,381.474352,5963.025148,0.122069,400.685697,3106.169924,0.015934,6.678095,51.769499,3.0,3.0,6.0
2,linear_svr,Linear Svr,390.971077,6346.064257,0.005658,375.756366,3127.956374,0.002082,6.262606,52.132606,7.0,2.0,9.0
3,linear_regression,Linear Regression,458.874439,6341.281110,0.007156,443.310053,3105.666212,0.016254,7.388501,51.761104,2.0,8.0,10.0
4,polynomial_regression_degree_2,Polynomial Regression Degree 2,442.933866,6337.248811,0.008418,428.393130,3123.216100,0.005104,7.139885,52.053602,5.0,6.0,11.0
5,hist_gradient_boosting,Hist Gradient Boosting,442.085917,6200.435428,0.050770,426.861429,3123.513120,0.004915,7.114357,52.058552,6.0,5.0,11.0
6,extra_trees,Extra Trees,458.200399,6169.761165,0.060139,451.392816,3117.071542,0.009015,7.523214,51.951192,4.0,9.0,13.0
7,decision_tree_depth_10,Decision Tree Depth 10,425.007217,6098.597210,0.081695,411.246417,3448.747367,-0.213099,6.854107,57.479123,10.0,4.0,14.0
8,decision_tree_depth_5,Decision Tree Depth 5,445.599676,6217.857860,0.045428,428.975570,3272.976549,-0.092595,7.149593,54.549609,9.0,7.0,16.0
9,baseline_dummy_median,Baseline Dummy Median,572.720409,6371.348834,-0.002282,555.208142,3144.172370,-0.008292,9.253469,52.402873,8.0,11.0,19.0



Best model by RMSE:
Hist Gradient Boosting Log Target
RMSE: 3076.37s | MAE: 300.96s

Best model by MAE:
Hist Gradient Boosting Log Target
MAE: 300.96s | RMSE: 3076.37s

Best model by combined RMSE + MAE rank:
Hist Gradient Boosting Log Target
MAE: 300.96s | RMSE: 3076.37s

Best regression model:
Selection rule: best combined RMSE + MAE rank
Model key: hist_gradient_boosting_log_target
Model name: Hist Gradient Boosting Log Target
Test MAE: 300.96 seconds (5.02 minutes)
Test RMSE: 3076.37 seconds (51.27 minutes)
Test R2: 0.0347

Test predictions sample:


,record_key,actual_elapsed_seconds,predicted_elapsed_seconds,absolute_error_seconds,actual_elapsed_minutes,predicted_elapsed_minutes,absolute_error_minutes
0,rid3437277,1104,800.958853,303.041147,18.400000,13.349314,5.050686
1,rid0012789,973,1004.044702,31.044702,16.216667,16.734078,0.517412
2,rid3431194,1730,1752.842335,22.842335,28.833333,29.214039,0.380706
3,rid0256066,626,749.948097,123.948097,10.433333,12.499135,2.065802
4,rid2874188,269,313.664329,44.664329,4.483333,5.227739,0.744405


Feature importance/coefficient export skipped:
name 'save_feature_importance_or_coefficients' is not defined

Saved files:
model_comparison.csv
test_prediction.csv
best_model.pkl
final_model_feature_columns.txt
modeling_report.txt

Phase 3 completed successfully.


In [65]:

expected_outputs = [    
    'model_comparison.csv',
    'test_predictions.csv',
    'modeling_report.txt',
    'best_model.pkl',
    'final_model_feature_columns.txt'
]

for  file in expected_outputs:
    print(file,'=>','CREATED' if Path(file).exists() else 'NOT FOUND')

if config.run_optional_classfication_demo:
    print('optional_duration_classfication_result.csv =>', 'CREATED' if Path('optional_duration_classification_results.csv').exists() else 'NOT FOUND')
    

model_comparison.csv => CREATED
test_predictions.csv => NOT FOUND
modeling_report.txt => CREATED
best_model.pkl => CREATED
final_model_feature_columns.txt => CREATED
